# MSc Artificial Intelligence Dissertation

## Notebook 02

# Feature Audit and Target Leakage Analysis

---

### Dissertation Title

Multi-Modal Credit Risk Prediction Using Structured and Textual Data:
Explainability, Calibration, and Decision Support

---

### Dataset

LendingClub Accepted Loans (2007–2018)

---

### Relationship to Previous Notebook

Notebook 01 established the overall characteristics and suitability of the LendingClub dataset for the proposed research.

Key findings from Notebook 01 included:

- the dataset contains 2,260,701 loan records and 151 variables;
- borrower-written loan descriptions are available for a subset of observations;
- completed loan outcomes, specifically Fully Paid and Charged Off, will form the supervised binary classification target;
- the available textual data is sufficiently large and diverse to support TF-IDF and BERT-based experiments.

This notebook builds on those findings by examining the temporal availability and modelling suitability of all dataset variables.

---

### Research Objective

To systematically classify LendingClub variables according to their availability at loan origination and identify features that can be used for predictive modelling without introducing target leakage.

---

### Research Question

Which LendingClub variables are genuinely available at or before the lending decision, and which variables must be excluded because they contain information generated after loan origination?

---

### Expected Outcome

By the end of this notebook:

- all 151 variables will be inventoried;
- variables will be classified by information timing and purpose;
- post-origination and leakage-prone features will be identified;
- ambiguous features will be reviewed separately;
- a defensible feature-selection strategy will be established for the preprocessing stage.

# Methodology

Feature selection will be conducted using a temporal-information framework.

Each variable will be reviewed and assigned to one of the following categories:

1. **Applicant-provided features**  
   Information supplied by the borrower during the application process.

2. **Lender-generated origination features**  
   Information produced by LendingClub during underwriting but available before the lending decision is completed.

3. **Administrative or identifier features**  
   Variables used for identification, record keeping, URLs, timestamps, or operational metadata rather than predictive risk assessment.

4. **Post-origination behavioural features**  
   Information generated after the loan has been issued, including repayment behaviour, outstanding balances, recoveries, collections, and payment history.

5. **Review-required features**  
   Variables whose availability or interpretation at prediction time requires explicit methodological justification.

Post-origination features will be excluded from predictive modelling because their inclusion would introduce target leakage and result in unrealistically optimistic model performance.

**Eligibility rule:** A predictor was considered eligible only when the underlying information could reasonably have been available at or before the lending decision. Variables generated from subsequent repayment, servicing, recovery, hardship, or settlement activity were treated as potential target leakage.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [3]:
dataset_path = (
    "/content/drive/MyDrive/Credit_Risk_Thesis/"
    "data/raw/accepted_2007_to_2018Q4.csv"
)

df = pd.read_csv(
    dataset_path,
    low_memory=False
)

print("Dataset shape:", df.shape)

Dataset shape: (2260701, 151)


## Discussion

The original LendingClub dataset is reloaded to ensure that the feature audit begins from the unmodified source data rather than from a previously filtered or transformed dataset.

This preserves methodological transparency and allows every feature-selection decision to be traced back to the original dataset.

## Key Findings

- The raw LendingClub dataset was loaded successfully.
- The dataset contains 2,260,701 observations and 151 variables.
- No feature filtering or preprocessing has been applied at this stage.

## Research Implication

All feature-selection decisions in this notebook will be based on the complete original feature set, reducing the risk that potentially important or leakage-prone variables are overlooked.

## Next Step

Create a complete inventory of the 151 variables before assigning them to methodological categories.

In [4]:
feature_inventory = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str).values
})

feature_inventory.head(10)

,feature,dtype
0,id,object
1,member_id,float64
2,loan_amnt,float64
3,funded_amnt,float64
4,funded_amnt_inv,float64
5,term,object
6,int_rate,float64
7,installment,float64
8,grade,object
9,sub_grade,object


In [5]:
print("Number of features:", len(feature_inventory))

Number of features: 151


In [6]:
pd.set_option("display.max_rows", None)
feature_inventory

,feature,dtype
0,id,object
1,member_id,float64
2,loan_amnt,float64
3,funded_amnt,float64
4,funded_amnt_inv,float64
5,term,object
6,int_rate,float64
7,installment,float64
8,grade,object
9,sub_grade,object


# 1. Feature Definition and Temporal Audit

## Purpose

The purpose of this section is to examine the meaning and temporal availability of each LendingClub features before deciding whether it is suitable for credit risk modelling.

Feature definitions are obtained from LendingClub-related data documentation. These definitions are used only to understand what each feature represents. The final classification of each feature as Keep, Remove, or Review is a methodological decision made in this research based on whether the information would legitimately be available at the intended prediction point.

For each feature, the following information will be recorded:

- **Definition:** documented meaning of the feature.
- **Information Timing:** when the information becomes available relative to loan origination.
- **Decision:** Keep, Remove, Review, or Target.
- **Justification:** research rationale for the decision.

In [7]:
feature_inventory["definition"] = ""
feature_inventory["information_timing"] = ""
feature_inventory["decision"] = ""
feature_inventory["justification"] = ""

feature_inventory.head(10)

,feature,dtype,definition,information_timing,decision,justification
0,id,object,,,,
1,member_id,float64,,,,
2,loan_amnt,float64,,,,
3,funded_amnt,float64,,,,
4,funded_amnt_inv,float64,,,,
5,term,object,,,,
6,int_rate,float64,,,,
7,installment,float64,,,,
8,grade,object,,,,
9,sub_grade,object,,,,


In [8]:
#  Let's audit first 10 features
first_10_audit = {
    "id": {
        "definition": "A unique LC assigned ID for the loan listing.",
        "information_timing": "Administrative / identifier",
        "decision": "Remove",
        "justification": "Unique identifier with no meaningful credit-risk interpretation and may encourage memorisation."
    },

    "member_id": {
        "definition": "A unique LC assigned ID for the borrower member.",
        "information_timing": "Administrative / identifier",
        "decision": "Remove",
        "justification": "Borrower identifier rather than an application-time risk characteristic."
    },

    "loan_amnt": {
        "definition": "The listed amount of the loan applied for by the borrower.",
        "information_timing": "Application / origination",
        "decision": "Keep",
        "justification": "Requested loan amount is known when the credit decision is made."
    },

    "funded_amnt": {
        "definition": "The total amount committed to that loan at that point in time.",
        "information_timing": "Funding / origination outcome",
        "decision": "Review",
        "justification": "Represents the amount ultimately committed rather than purely borrower-provided application information; its timing relative to the intended prediction point requires consideration."
    },

    "funded_amnt_inv": {
        "definition": "The total amount committed by investors for that loan at that point in time.",
        "information_timing": "Funding / origination outcome",
        "decision": "Review",
        "justification": "Reflects investor funding activity and may not be available at the intended application-time prediction point."
    },

    "term": {
        "definition": "The number of payments on the loan, expressed in months.",
        "information_timing": "Application / loan terms",
        "decision": "Keep",
        "justification": "The repayment term is established before loan performance is observed."
    },

    "int_rate": {
        "definition": "Interest rate on the loan.",
        "information_timing": "Lender underwriting / origination",
        "decision": "Review",
        "justification": "Available around origination but may already encode LendingClub's assessment of borrower risk."
    },

    "installment": {
        "definition": "The monthly payment owed by the borrower if the loan originates.",
        "information_timing": "Lender underwriting / origination",
        "decision": "Review",
        "justification": "Known before repayment begins but is derived from loan terms including amount, term and interest rate."
    },

    "grade": {
        "definition": "LC assigned loan grade.",
        "information_timing": "Lender underwriting / origination",
        "decision": "Review",
        "justification": "Not future-outcome leakage, but it directly incorporates LendingClub's existing credit-risk assessment."
    },

    "sub_grade": {
        "definition": "LC assigned loan subgrade.",
        "information_timing": "Lender underwriting / origination",
        "decision": "Review",
        "justification": "Provides a more granular LendingClub risk assessment and may therefore encode existing underwriting information."
    }
}

In [9]:
print(first_10_audit["loan_amnt"])
print(first_10_audit["loan_amnt"]["decision"])

{'definition': 'The listed amount of the loan applied for by the borrower.', 'information_timing': 'Application / origination', 'decision': 'Keep', 'justification': 'Requested loan amount is known when the credit decision is made.'}
Keep


In [10]:
# Put those audit information into our feature inventory table
for feature, audit in first_10_audit.items():

    mask = feature_inventory["feature"] == feature

    feature_inventory.loc[mask, "definition"] = audit["definition"]
    feature_inventory.loc[mask, "information_timing"] = audit["information_timing"]
    feature_inventory.loc[mask, "decision"] = audit["decision"]
    feature_inventory.loc[mask, "justification"] = audit["justification"]

In [11]:
feature_inventory.head(10)

,feature,dtype,definition,information_timing,decision,justification
0,id,object,A unique LC assigned ID for the loan listing.,Administrative / identifier,Remove,Unique identifier with no meaningful credit-ri...
1,member_id,float64,A unique LC assigned ID for the borrower member.,Administrative / identifier,Remove,Borrower identifier rather than an application...
2,loan_amnt,float64,The listed amount of the loan applied for by t...,Application / origination,Keep,Requested loan amount is known when the credit...
3,funded_amnt,float64,The total amount committed to that loan at tha...,Funding / origination outcome,Review,Represents the amount ultimately committed rat...
4,funded_amnt_inv,float64,The total amount committed by investors for th...,Funding / origination outcome,Review,Reflects investor funding activity and may not...
5,term,object,"The number of payments on the loan, expressed ...",Application / loan terms,Keep,The repayment term is established before loan ...
6,int_rate,float64,Interest rate on the loan.,Lender underwriting / origination,Review,Available around origination but may already e...
7,installment,float64,The monthly payment owed by the borrower if th...,Lender underwriting / origination,Review,Known before repayment begins but is derived f...
8,grade,object,LC assigned loan grade.,Lender underwriting / origination,Review,"Not future-outcome leakage, but it directly in..."
9,sub_grade,object,LC assigned loan subgrade.,Lender underwriting / origination,Review,Provides a more granular LendingClub risk asse...


### Interpretation

The first ten variables demonstrate that feature suitability cannot be determined solely from whether a variable exists before loan repayment.

Some variables, such as `loan_amnt` and `term`, represent information available at or before origination and are therefore suitable candidate predictors. Identifier variables such as `id` and `member_id` do not represent borrower credit risk and are excluded.

Other variables, including `int_rate`, `grade`, and `sub_grade`, require further consideration. Although these variables are available around loan origination and therefore do not constitute conventional post-outcome target leakage, they incorporate information generated by LendingClub's underwriting process. Their inclusion could therefore cause the proposed model to partially reproduce LendingClub's existing risk assessment rather than independently estimate credit risk.

Variables classified as **Review** will be reconsidered when the final predictor set and ablation experiments are defined.

### Key Findings

- Feature eligibility depends on both temporal availability and the role of the variable in the lending process.
- Applicant and credit information available before the lending decision can potentially be retained.
- Unique identifiers should be excluded from predictive modelling.
- Post-origination information will be excluded because it can introduce target leakage.
- Lender-generated underwriting variables require separate consideration because they may encode an existing risk assessment.

# 2. Employment, Income, Target and Text Feature Audit

This group contains borrower employment and income characteristics, LendingClub-generated verification information, temporal metadata, the target variable, administrative information, and the borrower-written loan description.

Particular attention is given to `loan_status`, which defines the prediction target, and `desc`, which provides the principal unstructured information required for the hybrid NLP experiments.

In [12]:
features_11_20_audit = {

    "emp_title": {
        "definition": "The job title supplied by the borrower when applying for the loan.",
        "information_timing": "Application / borrower-provided",
        "decision": "Review",
        "justification": "Available at application time, but the variable is likely to have high cardinality, inconsistent wording and substantial missingness. Its modelling value should therefore be assessed before inclusion."
    },

    "emp_length": {
        "definition": "Employment length in years, with values ranging from less than one year to ten or more years.",
        "information_timing": "Application / borrower-provided",
        "decision": "Keep",
        "justification": "Employment history is available during application and may provide information about borrower financial stability."
    },

    "home_ownership": {
        "definition": "Home ownership status provided by the borrower during registration or obtained from the credit report.",
        "information_timing": "Application / borrower-credit information",
        "decision": "Keep",
        "justification": "Available before the loan outcome and potentially informative of the borrower's financial circumstances."
    },

    "annual_inc": {
        "definition": "Self-reported annual income provided by the borrower during registration.",
        "information_timing": "Application / borrower-provided",
        "decision": "Keep",
        "justification": "Income is available during underwriting and is directly relevant to the borrower's capacity to service debt."
    },

    "verification_status": {
        "definition": "Indicates whether the borrower's income was verified, source verified, or not verified by LendingClub.",
        "information_timing": "Underwriting / pre-origination",
        "decision": "Keep",
        "justification": "Income verification status is established during underwriting and is available before subsequent repayment behaviour is observed."
    },

    "issue_d": {
        "definition": "The month in which the loan was funded.",
        "information_timing": "Origination / temporal metadata",
        "decision": "Review",
        "justification": "The funding month should not automatically be treated as a borrower risk characteristic, but it may be important for temporal cohort analysis and chronological train-test splitting."
    },

    "loan_status": {
        "definition": "Current status of the loan.",
        "information_timing": "Outcome / target",
        "decision": "Target",
        "justification": "This variable is used to construct the binary outcome (Fully Paid versus Charged Off) and must never be included among the predictor variables."
    },

    "pymnt_plan": {
        "definition": "Indicates whether a payment plan has been put in place for the loan.",
        "information_timing": "Potentially post-origination / servicing",
        "decision": "Review",
        "justification": "The timing and meaning of the payment-plan indicator require verification. If it reflects a servicing arrangement created after origination, it must be excluded as leakage."
    },

    "url": {
        "definition": "URL for the LendingClub page containing data associated with the loan.",
        "information_timing": "Administrative / identifier",
        "decision": "Remove",
        "justification": "The URL is administrative metadata and does not represent a legitimate borrower credit-risk characteristic."
    },

    "desc": {
        "definition": "Loan description provided by the borrower.",
        "information_timing": "Application / borrower-provided text",
        "decision": "Keep",
        "justification": "The description is available during the application process and is the primary unstructured text variable required for the TF-IDF and BERT components of this research."
    }
}

In [13]:
for feature, audit in features_11_20_audit.items():

    mask = feature_inventory["feature"] == feature

    feature_inventory.loc[mask, "definition"] = audit["definition"]
    feature_inventory.loc[mask, "information_timing"] = audit["information_timing"]
    feature_inventory.loc[mask, "decision"] = audit["decision"]
    feature_inventory.loc[mask, "justification"] = audit["justification"]

In [14]:
feature_inventory.iloc[10:20]

,feature,dtype,definition,information_timing,decision,justification
10,emp_title,object,The job title supplied by the borrower when ap...,Application / borrower-provided,Review,"Available at application time, but the variabl..."
11,emp_length,object,"Employment length in years, with values rangin...",Application / borrower-provided,Keep,Employment history is available during applica...
12,home_ownership,object,Home ownership status provided by the borrower...,Application / borrower-credit information,Keep,Available before the loan outcome and potentia...
13,annual_inc,float64,Self-reported annual income provided by the bo...,Application / borrower-provided,Keep,Income is available during underwriting and is...
14,verification_status,object,Indicates whether the borrower's income was ve...,Underwriting / pre-origination,Keep,Income verification status is established duri...
15,issue_d,object,The month in which the loan was funded.,Origination / temporal metadata,Review,The funding month should not automatically be ...
16,loan_status,object,Current status of the loan.,Outcome / target,Target,This variable is used to construct the binary ...
17,pymnt_plan,object,Indicates whether a payment plan has been put ...,Potentially post-origination / servicing,Review,The timing and meaning of the payment-plan ind...
18,url,object,URL for the LendingClub page containing data a...,Administrative / identifier,Remove,The URL is administrative metadata and does no...
19,desc,object,Loan description provided by the borrower.,Application / borrower-provided text,Keep,The description is available during the applic...


In [15]:
# Check audit progress

audited = feature_inventory["decision"].ne("").sum()
remaining = feature_inventory["decision"].eq("").sum()

print("Total features :", len(feature_inventory))
print("Audited        :", audited)
print("Remaining      :", remaining)

Total features : 151
Audited        : 20
Remaining      : 131


In [16]:
feature_inventory.iloc[:20]["decision"].value_counts()

,count
decision,
Review,9
Keep,7
Remove,3
Target,1


# 3. Application and Credit-Bureau Feature Audit

This group primarily contains information describing the purpose of the loan, borrower location, indebtedness, credit history, FICO score, credit enquiries, public records, revolving credit utilisation, and account history.

These variables are particularly important for the structured component of the proposed hybrid credit-risk model. Each variable is assessed according to whether the information is available at or before loan origination and whether its inclusion is methodologically appropriate.

In [17]:
features_21_37_audit = {

    "purpose": {
        "definition": "A category provided by the borrower for the loan request.",
        "information_timing": "Application / borrower-provided",
        "decision": "Keep",
        "justification": "Loan purpose is known during application and may contain information relevant to credit risk."
    },

    "title": {
        "definition": "The loan title provided by the borrower.",
        "information_timing": "Application / borrower-provided text",
        "decision": "Review",
        "justification": "Available at application, but it may overlap substantially with purpose and the borrower description. Its incremental predictive value should be assessed before inclusion."
    },

    "zip_code": {
        "definition": "The first three numbers of the zip code provided by the borrower in the loan application.",
        "information_timing": "Application / geographic",
        "decision": "Review",
        "justification": "Available at application but introduces geographic information that may create high cardinality, proxy effects, and limited generalisability."
    },

    "addr_state": {
        "definition": "The state provided by the borrower in the loan application.",
        "information_timing": "Application / geographic",
        "decision": "Review",
        "justification": "Available at application, but geographic information requires justification because it may capture regional effects rather than borrower-specific creditworthiness."
    },

    "dti": {
        "definition": "Ratio of the borrower's total monthly debt payments to monthly income.",
        "information_timing": "Application / financial",
        "decision": "Keep",
        "justification": "Debt-to-income ratio is available during underwriting and directly represents the borrower's existing debt burden relative to income."
    },

    "delinq_2yrs": {
        "definition": "Number of 30+ days past-due incidences of delinquency in the borrower's credit file during the past two years.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Represents historical credit behaviour observable before the new loan outcome."
    },

    "earliest_cr_line": {
        "definition": "Month in which the borrower's earliest reported credit line was opened.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Available from the credit file before origination and can be transformed into credit-history length."
    },

    "fico_range_low": {
        "definition": "Lower boundary of the borrower's FICO score range at loan origination.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Represents credit-risk information available at or around underwriting before subsequent repayment behaviour occurs."
    },

    "fico_range_high": {
        "definition": "Upper boundary of the borrower's FICO score range at loan origination.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Represents credit-risk information available at or around underwriting before subsequent repayment behaviour occurs."
    },

    "inq_last_6mths": {
        "definition": "Number of credit enquiries during the previous six months.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Recent credit enquiry activity is observable before origination and may indicate credit-seeking behaviour."
    },

    "mths_since_last_delinq": {
        "definition": "Number of months since the borrower's most recent delinquency.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Historical delinquency recency is available from the credit report before the new loan outcome."
    },

    "mths_since_last_record": {
        "definition": "Number of months since the borrower's most recent public record.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Historical public-record information is available before loan origination."
    },

    "open_acc": {
        "definition": "Number of open credit lines in the borrower's credit file.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Current open-account information is available during underwriting."
    },

    "pub_rec": {
        "definition": "Number of derogatory public records.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing derogatory public records are observable before the loan decision and may provide legitimate risk information."
    },

    "revol_bal": {
        "definition": "Total revolving credit balance.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing revolving debt is observable before origination and provides information about borrower indebtedness."
    },

    "revol_util": {
        "definition": "Revolving line utilisation rate, representing the amount of credit being used relative to available revolving credit.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Credit utilisation is observable during underwriting and is a meaningful indicator of existing credit usage."
    },

    "total_acc": {
        "definition": "Total number of credit lines currently in the borrower's credit file.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Represents the borrower's existing credit history and is available before loan performance is observed."
    },

    "initial_list_status": {
        "definition": "Initial listing status of the loan.",
        "information_timing": "Origination / platform-generated",
        "decision": "Review",
        "justification": "This is LendingClub platform metadata rather than a direct borrower-risk characteristic. Its relevance and timing should be assessed before inclusion."
    }
}

In [18]:
for feature, audit in features_21_37_audit.items():

    mask = feature_inventory["feature"] == feature

    feature_inventory.loc[mask, "definition"] = audit["definition"]
    feature_inventory.loc[mask, "information_timing"] = audit["information_timing"]
    feature_inventory.loc[mask, "decision"] = audit["decision"]
    feature_inventory.loc[mask, "justification"] = audit["justification"]

In [19]:
feature_inventory.iloc[20:38]

,feature,dtype,definition,information_timing,decision,justification
20,purpose,object,A category provided by the borrower for the lo...,Application / borrower-provided,Keep,Loan purpose is known during application and m...
21,title,object,The loan title provided by the borrower.,Application / borrower-provided text,Review,"Available at application, but it may overlap s..."
22,zip_code,object,The first three numbers of the zip code provid...,Application / geographic,Review,Available at application but introduces geogra...
23,addr_state,object,The state provided by the borrower in the loan...,Application / geographic,Review,"Available at application, but geographic infor..."
24,dti,float64,Ratio of the borrower's total monthly debt pay...,Application / financial,Keep,Debt-to-income ratio is available during under...
25,delinq_2yrs,float64,Number of 30+ days past-due incidences of deli...,Pre-origination / credit bureau,Keep,Represents historical credit behaviour observa...
26,earliest_cr_line,object,Month in which the borrower's earliest reporte...,Pre-origination / credit bureau,Keep,Available from the credit file before originat...
27,fico_range_low,float64,Lower boundary of the borrower's FICO score ra...,Pre-origination / credit bureau,Keep,Represents credit-risk information available a...
28,fico_range_high,float64,Upper boundary of the borrower's FICO score ra...,Pre-origination / credit bureau,Keep,Represents credit-risk information available a...
29,inq_last_6mths,float64,Number of credit enquiries during the previous...,Pre-origination / credit bureau,Keep,Recent credit enquiry activity is observable b...


In [20]:
audited = feature_inventory["decision"].ne("").sum()
remaining = feature_inventory["decision"].eq("").sum()

print("Total features :", len(feature_inventory))
print("Audited        :", audited)
print("Remaining      :", remaining)

Total features : 151
Audited        : 38
Remaining      : 113


# 4. Post-Origination Target Leakage Analysis

## Purpose

This section identifies variables that are generated after the loan has been issued or after repayment behaviour has begun.

Such variables cannot legitimately be used to predict default at the time of underwriting because they reveal information about the future loan outcome.

Including these variables would introduce target leakage and could result in unrealistically high predictive performance.

The objective of this section is therefore to identify and exclude repayment, recovery, servicing, and post-origination credit information from the modelling feature set.

In [21]:
features_38_52_audit = {

    "out_prncp": {
        "definition": "Remaining outstanding principal for the total amount funded.",
        "information_timing": "Post-origination / repayment",
        "decision": "Remove",
        "justification": "Outstanding principal is only observed after the loan has been issued and repayments have begun. It directly reflects subsequent loan performance."
    },

    "out_prncp_inv": {
        "definition": "Remaining outstanding principal for the portion of the loan funded by investors.",
        "information_timing": "Post-origination / repayment",
        "decision": "Remove",
        "justification": "Represents the remaining investor principal after origination and therefore contains future repayment information."
    },

    "total_pymnt": {
        "definition": "Total payments received to date for the funded amount.",
        "information_timing": "Post-origination / repayment",
        "decision": "Remove",
        "justification": "Total payments are only known after repayment begins and strongly reveal whether a loan is performing."
    },

    "total_pymnt_inv": {
        "definition": "Total payments received to date for the investor-funded portion of the loan.",
        "information_timing": "Post-origination / repayment",
        "decision": "Remove",
        "justification": "Contains realised repayment information unavailable at the underwriting stage."
    },

    "total_rec_prncp": {
        "definition": "Principal amount received to date.",
        "information_timing": "Post-origination / repayment",
        "decision": "Remove",
        "justification": "Represents realised repayment behaviour and therefore directly leaks information about the outcome."
    },

    "total_rec_int": {
        "definition": "Interest received to date.",
        "information_timing": "Post-origination / repayment",
        "decision": "Remove",
        "justification": "Interest received is generated only after loan repayment begins."
    },

    "total_rec_late_fee": {
        "definition": "Late fees received to date.",
        "information_timing": "Post-origination / repayment",
        "decision": "Remove",
        "justification": "Late-fee information directly indicates subsequent delinquency behaviour and therefore constitutes severe target leakage."
    },

    "recoveries": {
        "definition": "Post-charge-off gross recovery amount.",
        "information_timing": "Post-default / recovery",
        "decision": "Remove",
        "justification": "Recovery information is generated only after severe default or charge-off and therefore directly reveals the target outcome."
    },

    "collection_recovery_fee": {
        "definition": "Collection fee associated with post-charge-off recovery.",
        "information_timing": "Post-default / recovery",
        "decision": "Remove",
        "justification": "Only exists after collection or recovery activity and therefore represents direct outcome leakage."
    },

    "last_pymnt_d": {
        "definition": "Month in which the most recent payment was received.",
        "information_timing": "Post-origination / repayment",
        "decision": "Remove",
        "justification": "The most recent payment date occurs after loan issuance and provides direct information about repayment behaviour."
    },

    "last_pymnt_amnt": {
        "definition": "Most recent total payment amount received.",
        "information_timing": "Post-origination / repayment",
        "decision": "Remove",
        "justification": "Payment amount is observed only after origination and therefore cannot be used for application-time risk prediction."
    },

    "next_pymnt_d": {
        "definition": "Next scheduled payment date.",
        "information_timing": "Post-origination / servicing",
        "decision": "Remove",
        "justification": "Represents servicing information generated after the loan has been issued."
    },

    "last_credit_pull_d": {
        "definition": "Most recent month in which LendingClub obtained the borrower's credit information.",
        "information_timing": "Potentially post-origination / monitoring",
        "decision": "Remove",
        "justification": "The latest credit pull may occur after loan origination and therefore does not reliably represent information available at the initial lending decision."
    },

    "last_fico_range_high": {
        "definition": "Upper boundary of the borrower's most recently observed FICO score range.",
        "information_timing": "Post-origination / monitoring",
        "decision": "Remove",
        "justification": "Represents an updated FICO score observed after origination and therefore contains future borrower information."
    },

    "last_fico_range_low": {
        "definition": "Lower boundary of the borrower's most recently observed FICO score range.",
        "information_timing": "Post-origination / monitoring",
        "decision": "Remove",
        "justification": "Represents an updated FICO score observed after origination and therefore constitutes future information."
    }
}

In [22]:
for feature, audit in features_38_52_audit.items():

    mask = feature_inventory["feature"] == feature

    feature_inventory.loc[mask, "definition"] = audit["definition"]
    feature_inventory.loc[mask, "information_timing"] = audit["information_timing"]
    feature_inventory.loc[mask, "decision"] = audit["decision"]
    feature_inventory.loc[mask, "justification"] = audit["justification"]

In [23]:
feature_inventory.iloc[38:53]

,feature,dtype,definition,information_timing,decision,justification
38,out_prncp,float64,Remaining outstanding principal for the total ...,Post-origination / repayment,Remove,Outstanding principal is only observed after t...
39,out_prncp_inv,float64,Remaining outstanding principal for the portio...,Post-origination / repayment,Remove,Represents the remaining investor principal af...
40,total_pymnt,float64,Total payments received to date for the funded...,Post-origination / repayment,Remove,Total payments are only known after repayment ...
41,total_pymnt_inv,float64,Total payments received to date for the invest...,Post-origination / repayment,Remove,Contains realised repayment information unavai...
42,total_rec_prncp,float64,Principal amount received to date.,Post-origination / repayment,Remove,Represents realised repayment behaviour and th...
43,total_rec_int,float64,Interest received to date.,Post-origination / repayment,Remove,Interest received is generated only after loan...
44,total_rec_late_fee,float64,Late fees received to date.,Post-origination / repayment,Remove,Late-fee information directly indicates subseq...
45,recoveries,float64,Post-charge-off gross recovery amount.,Post-default / recovery,Remove,Recovery information is generated only after s...
46,collection_recovery_fee,float64,Collection fee associated with post-charge-off...,Post-default / recovery,Remove,Only exists after collection or recovery activ...
47,last_pymnt_d,object,Month in which the most recent payment was rec...,Post-origination / repayment,Remove,The most recent payment date occurs after loan...


In [24]:
audited = feature_inventory["decision"].ne("").sum()
remaining = feature_inventory["decision"].eq("").sum()

print("Total features :", len(feature_inventory))
print("Audited        :", audited)
print("Remaining      :", remaining)

Total features : 151
Audited        : 53
Remaining      : 98


## Discussion

The variables audited in this section are generated after loan origination and primarily describe realised repayment behaviour, servicing activity, updated borrower credit information, or recovery activity.

These variables would not have been available when the original lending decision was made. Their inclusion in the predictor set would therefore allow the model to access information about the future outcome it is supposed to predict.

The strongest examples of direct target leakage include `total_pymnt`, `recoveries`, `collection_recovery_fee`, and the most recent payment and FICO variables.

## Key Findings

- All variables in this block contain post-origination or post-default information.
- Payment and recovery variables directly reveal subsequent loan performance.
- Updated FICO scores represent future borrower information rather than origination-time creditworthiness.
- These features must be excluded from the predictor matrix.

## Research Implication

Removing these features is necessary to ensure that model performance represents genuine application-time credit risk prediction rather than retrospective classification using future repayment information.

This leakage-control step applies equally to the structured-only, TF-IDF hybrid, and BERT hybrid models so that all model families are evaluated under the same information constraints.

In [25]:
feature_inventory.iloc[53:78][["feature", "dtype"]]

,feature,dtype
53,collections_12_mths_ex_med,float64
54,mths_since_last_major_derog,float64
55,policy_code,float64
56,application_type,object
57,annual_inc_joint,float64
58,dti_joint,float64
59,verification_status_joint,object
60,acc_now_delinq,float64
61,tot_coll_amt,float64
62,tot_cur_bal,float64


# 5. Extended Credit-Bureau and Joint-Application Feature Audit

## Purpose

This section evaluates additional credit-bureau attributes and joint-application variables.

Unlike the post-origination variables examined in the previous section, most variables in this group describe the borrower's existing credit position, account activity, utilisation, enquiries, and credit history at or around the time of underwriting.

Joint-application variables are considered separately because their relevance depends on whether the loan application was submitted individually or jointly.

In [26]:
features_53_77_audit = {

    "collections_12_mths_ex_med": {
        "definition": "Number of collections recorded during the previous 12 months, excluding medical collections.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Historical collection activity is observable before the new loan outcome and may provide legitimate credit-risk information."
    },

    "mths_since_last_major_derog": {
        "definition": "Number of months since the borrower's most recent major derogatory credit event.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Represents historical adverse credit behaviour available during underwriting."
    },

    "policy_code": {
        "definition": "Publicly available policy-code indicator associated with the loan.",
        "information_timing": "Platform / administrative",
        "decision": "Review",
        "justification": "This is LendingClub policy metadata rather than a borrower-risk characteristic. Its distribution and informational value should be examined before final exclusion."
    },

    "application_type": {
        "definition": "Indicates whether the loan application is individual or joint.",
        "information_timing": "Application / application structure",
        "decision": "Keep",
        "justification": "Application type is known during underwriting and is required to correctly interpret joint-applicant variables."
    },

    "annual_inc_joint": {
        "definition": "Combined self-reported annual income of the co-borrowers for a joint application.",
        "information_timing": "Application / joint borrower-provided",
        "decision": "Keep",
        "justification": "Available at application for joint loans and provides relevant information about combined repayment capacity."
    },

    "dti_joint": {
        "definition": "Debt-to-income ratio calculated using the combined monthly debt payments and income of joint applicants.",
        "information_timing": "Application / joint financial",
        "decision": "Keep",
        "justification": "Represents joint-applicant indebtedness available during underwriting."
    },

    "verification_status_joint": {
        "definition": "Indicates the income-verification status for a joint application.",
        "information_timing": "Underwriting / joint application",
        "decision": "Keep",
        "justification": "Joint income verification information is established during underwriting before loan performance is observed."
    },

    "acc_now_delinq": {
        "definition": "Number of accounts on which the borrower is currently delinquent.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Current delinquency status from the credit file is available during underwriting and is directly relevant to credit risk."
    },

    "tot_coll_amt": {
        "definition": "Total collection amounts recorded in the borrower's credit file.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing collection balances are historical credit information available before the new loan outcome."
    },

    "tot_cur_bal": {
        "definition": "Total current balance across the borrower's accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing account balances are observable from the credit report during underwriting."
    },

    "open_acc_6m": {
        "definition": "Number of open credit accounts opened during the previous six months.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Recent account-opening activity is available before origination and may capture recent credit-seeking behaviour."
    },

    "open_act_il": {
        "definition": "Number of currently active installment accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing active installment obligations are available during underwriting."
    },

    "open_il_12m": {
        "definition": "Number of installment accounts opened during the previous 12 months.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Recent installment-credit activity is observable before the new loan is originated."
    },

    "open_il_24m": {
        "definition": "Number of installment accounts opened during the previous 24 months.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Historical installment-account opening activity is available during underwriting."
    },

    "mths_since_rcnt_il": {
        "definition": "Number of months since the most recent installment account was opened.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "The recency of installment-credit activity is known from the credit report before loan performance occurs."
    },

    "total_bal_il": {
        "definition": "Total current balance of installment accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing installment debt is available during underwriting and represents current borrower indebtedness."
    },

    "il_util": {
        "definition": "Installment-account utilisation ratio.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing installment-credit utilisation is observable before origination."
    },

    "open_rv_12m": {
        "definition": "Number of revolving accounts opened during the previous 12 months.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Recent revolving-credit activity is observable from the credit report before the lending outcome."
    },

    "open_rv_24m": {
        "definition": "Number of revolving accounts opened during the previous 24 months.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Historical revolving-account opening activity is available during underwriting."
    },

    "max_bal_bc": {
        "definition": "Maximum current balance owed on bankcard accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing bankcard balance information is available from the borrower's credit file."
    },

    "all_util": {
        "definition": "Balance-to-credit-limit utilisation across credit accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing credit utilisation is observable during underwriting and may indicate borrower leverage."
    },

    "total_rev_hi_lim": {
        "definition": "Total revolving high-credit or credit-limit amount.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing revolving credit limits are available from the credit report before origination."
    },

    "inq_fi": {
        "definition": "Number of personal-finance credit enquiries.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Historical finance enquiries are observable before loan origination and may indicate recent credit-seeking behaviour."
    },

    "total_cu_tl": {
        "definition": "Number of finance-related credit accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing finance-account history is available during underwriting."
    },

    "inq_last_12m": {
        "definition": "Number of credit enquiries during the previous 12 months.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Recent enquiry activity is known before origination and may indicate increased demand for credit."
    }
}

In [27]:
for feature, audit in features_53_77_audit.items():

    mask = feature_inventory["feature"] == feature

    feature_inventory.loc[mask, "definition"] = audit["definition"]
    feature_inventory.loc[mask, "information_timing"] = audit["information_timing"]
    feature_inventory.loc[mask, "decision"] = audit["decision"]
    feature_inventory.loc[mask, "justification"] = audit["justification"]

In [28]:
feature_inventory.iloc[53:78]

,feature,dtype,definition,information_timing,decision,justification
53,collections_12_mths_ex_med,float64,Number of collections recorded during the prev...,Pre-origination / credit bureau,Keep,Historical collection activity is observable b...
54,mths_since_last_major_derog,float64,Number of months since the borrower's most rec...,Pre-origination / credit bureau,Keep,Represents historical adverse credit behaviour...
55,policy_code,float64,Publicly available policy-code indicator assoc...,Platform / administrative,Review,This is LendingClub policy metadata rather tha...
56,application_type,object,Indicates whether the loan application is indi...,Application / application structure,Keep,Application type is known during underwriting ...
57,annual_inc_joint,float64,Combined self-reported annual income of the co...,Application / joint borrower-provided,Keep,Available at application for joint loans and p...
58,dti_joint,float64,Debt-to-income ratio calculated using the comb...,Application / joint financial,Keep,Represents joint-applicant indebtedness availa...
59,verification_status_joint,object,Indicates the income-verification status for a...,Underwriting / joint application,Keep,Joint income verification information is estab...
60,acc_now_delinq,float64,Number of accounts on which the borrower is cu...,Pre-origination / credit bureau,Keep,Current delinquency status from the credit fil...
61,tot_coll_amt,float64,Total collection amounts recorded in the borro...,Pre-origination / credit bureau,Keep,Existing collection balances are historical cr...
62,tot_cur_bal,float64,Total current balance across the borrower's ac...,Pre-origination / credit bureau,Keep,Existing account balances are observable from ...


In [29]:
audited = feature_inventory["decision"].ne("").sum()
remaining = feature_inventory["decision"].eq("").sum()

print("Total features :", len(feature_inventory))
print("Audited        :", audited)
print("Remaining      :", remaining)

Total features : 151
Audited        : 78
Remaining      : 73


## 5.1 Detailed Credit-History Feature Audit

This section evaluates detailed credit-bureau attributes describing account activity, balances, utilisation, delinquency history, credit limits, bankruptcies, liens, and the age and composition of the borrower's credit portfolio.

These variables are assessed according to the same temporal criterion used throughout the feature audit: whether the information represents the borrower's credit position observable at or before loan origination rather than subsequent performance of the LendingClub loan.

In [30]:
features_78_114_audit = {

    "acc_open_past_24mths": {
        "definition": "Number of credit accounts opened during the past 24 months.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Historical account-opening activity is observable before loan origination."
    },

    "avg_cur_bal": {
        "definition": "Average current balance across credit accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing account balances are observable during underwriting."
    },

    "bc_open_to_buy": {
        "definition": "Total available credit on revolving bankcard accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Available bankcard credit is part of the borrower's existing credit position."
    },

    "bc_util": {
        "definition": "Ratio of current bankcard balances to bankcard credit limits.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Bankcard utilisation is observable before origination and may indicate credit pressure."
    },

    "chargeoff_within_12_mths": {
        "definition": "Number of charge-offs recorded during the previous 12 months.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "This describes historical charge-offs occurring before the LendingClub loan, rather than the outcome of the loan being predicted."
    },

    "delinq_amnt": {
        "definition": "Amount currently past due on delinquent accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing delinquent debt is observable during underwriting."
    },

    "mo_sin_old_il_acct": {
        "definition": "Number of months since the oldest installment account was opened.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Represents historical installment-credit age available before origination."
    },

    "mo_sin_old_rev_tl_op": {
        "definition": "Number of months since the oldest revolving account was opened.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Represents the age of the borrower's revolving-credit history."
    },

    "mo_sin_rcnt_rev_tl_op": {
        "definition": "Number of months since the most recent revolving account was opened.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Recent revolving-account activity is observable from the credit report."
    },

    "mo_sin_rcnt_tl": {
        "definition": "Number of months since the most recent credit account was opened.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Recent account-opening activity is known before loan origination."
    },

    "mort_acc": {
        "definition": "Number of mortgage accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Mortgage-account history is observable from the borrower's credit report."
    },

    "mths_since_recent_bc": {
        "definition": "Number of months since the most recent bankcard account was opened.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Bankcard account recency is observable before origination."
    },

    "mths_since_recent_bc_dlq": {
        "definition": "Number of months since the most recent bankcard delinquency.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Historical bankcard delinquency is available before the LendingClub loan outcome."
    },

    "mths_since_recent_inq": {
        "definition": "Number of months since the most recent credit enquiry.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Credit-enquiry recency is known at underwriting."
    },

    "mths_since_recent_revol_delinq": {
        "definition": "Number of months since the most recent revolving-credit delinquency.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Historical revolving delinquency is observable before origination."
    },

    "num_accts_ever_120_pd": {
        "definition": "Number of accounts that have ever been 120 or more days past due.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Represents historical severe delinquency before the new loan."
    },

    "num_actv_bc_tl": {
        "definition": "Number of currently active bankcard accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Current bankcard activity is observable during underwriting."
    },

    "num_actv_rev_tl": {
        "definition": "Number of currently active revolving accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Current revolving-credit activity is available before origination."
    },

    "num_bc_sats": {
        "definition": "Number of satisfactory bankcard accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing bankcard account status is available from the credit report."
    },

    "num_bc_tl": {
        "definition": "Number of bankcard accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Bankcard-account history exists before the new loan outcome."
    },

    "num_il_tl": {
        "definition": "Number of installment accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Installment-account history is observable during underwriting."
    },

    "num_op_rev_tl": {
        "definition": "Number of open revolving accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Open revolving accounts are part of the borrower's existing credit profile."
    },

    "num_rev_accts": {
        "definition": "Number of revolving accounts in the borrower's credit history.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Revolving-account history is observable before loan origination."
    },

    "num_rev_tl_bal_gt_0": {
        "definition": "Number of revolving accounts with a balance greater than zero.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing revolving balances are available during underwriting."
    },

    "num_sats": {
        "definition": "Number of satisfactory credit accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing account status is observable before the LendingClub loan outcome."
    },

    "num_tl_120dpd_2m": {
        "definition": "Number of accounts currently 120 or more days past due, updated within the previous two months.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Current severe delinquency in existing accounts is available from the credit report at underwriting."
    },

    "num_tl_30dpd": {
        "definition": "Number of accounts currently 30 days past due.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Current delinquency on existing credit accounts is observable before origination."
    },

    "num_tl_90g_dpd_24m": {
        "definition": "Number of accounts 90 or more days past due during the previous 24 months.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Historical severe delinquency is known before the new loan is issued."
    },

    "num_tl_op_past_12m": {
        "definition": "Number of credit accounts opened during the previous 12 months.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Recent account-opening activity is observable at underwriting."
    },

    "pct_tl_nvr_dlq": {
        "definition": "Percentage of credit accounts that have never been delinquent.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Summarises historical repayment behaviour on existing accounts before origination."
    },

    "percent_bc_gt_75": {
        "definition": "Percentage of bankcard accounts with utilisation greater than 75 percent.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "High utilisation of existing bankcards is observable during underwriting."
    },

    "pub_rec_bankruptcies": {
        "definition": "Number of public-record bankruptcies.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing bankruptcy records are known before the loan decision."
    },

    "tax_liens": {
        "definition": "Number of tax liens.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing tax liens are observable from the credit record before origination."
    },

    "tot_hi_cred_lim": {
        "definition": "Total high-credit or credit-limit amount across accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing total credit limits are available during underwriting."
    },

    "total_bal_ex_mort": {
        "definition": "Total credit balance excluding mortgage balances.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing non-mortgage debt is observable before origination."
    },

    "total_bc_limit": {
        "definition": "Total bankcard credit limit.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing bankcard limits are available from the credit report."
    },

    "total_il_high_credit_limit": {
        "definition": "Total high-credit or credit-limit amount for installment accounts.",
        "information_timing": "Pre-origination / credit bureau",
        "decision": "Keep",
        "justification": "Existing installment-credit limits are observable during underwriting."
    }
}

In [31]:
for feature, audit in features_78_114_audit.items():

    mask = feature_inventory["feature"] == feature

    feature_inventory.loc[mask, "definition"] = audit["definition"]
    feature_inventory.loc[mask, "information_timing"] = audit["information_timing"]
    feature_inventory.loc[mask, "decision"] = audit["decision"]
    feature_inventory.loc[mask, "justification"] = audit["justification"]

In [32]:
feature_inventory.iloc[78:115]

,feature,dtype,definition,information_timing,decision,justification
78,acc_open_past_24mths,float64,Number of credit accounts opened during the pa...,Pre-origination / credit bureau,Keep,Historical account-opening activity is observa...
79,avg_cur_bal,float64,Average current balance across credit accounts.,Pre-origination / credit bureau,Keep,Existing account balances are observable durin...
80,bc_open_to_buy,float64,Total available credit on revolving bankcard a...,Pre-origination / credit bureau,Keep,Available bankcard credit is part of the borro...
81,bc_util,float64,Ratio of current bankcard balances to bankcard...,Pre-origination / credit bureau,Keep,Bankcard utilisation is observable before orig...
82,chargeoff_within_12_mths,float64,Number of charge-offs recorded during the prev...,Pre-origination / credit bureau,Keep,This describes historical charge-offs occurrin...
83,delinq_amnt,float64,Amount currently past due on delinquent accounts.,Pre-origination / credit bureau,Keep,Existing delinquent debt is observable during ...
84,mo_sin_old_il_acct,float64,Number of months since the oldest installment ...,Pre-origination / credit bureau,Keep,Represents historical installment-credit age a...
85,mo_sin_old_rev_tl_op,float64,Number of months since the oldest revolving ac...,Pre-origination / credit bureau,Keep,Represents the age of the borrower's revolving...
86,mo_sin_rcnt_rev_tl_op,float64,Number of months since the most recent revolvi...,Pre-origination / credit bureau,Keep,Recent revolving-account activity is observabl...
87,mo_sin_rcnt_tl,float64,Number of months since the most recent credit ...,Pre-origination / credit bureau,Keep,Recent account-opening activity is known befor...


In [33]:
audited = feature_inventory["decision"].ne("").sum()
remaining = feature_inventory["decision"].eq("").sum()

print("Total features :", len(feature_inventory))
print("Audited        :", audited)
print("Remaining      :", remaining)

Total features : 151
Audited        : 115
Remaining      : 36


In [34]:
feature_inventory.iloc[78:115][
    ["feature", "decision"]
]

,feature,decision
78,acc_open_past_24mths,Keep
79,avg_cur_bal,Keep
80,bc_open_to_buy,Keep
81,bc_util,Keep
82,chargeoff_within_12_mths,Keep
83,delinq_amnt,Keep
84,mo_sin_old_il_acct,Keep
85,mo_sin_old_rev_tl_op,Keep
86,mo_sin_rcnt_rev_tl_op,Keep
87,mo_sin_rcnt_tl,Keep


In [35]:
feature_inventory.iloc[115:128][["feature", "dtype"]]

,feature,dtype
115,revol_bal_joint,float64
116,sec_app_fico_range_low,float64
117,sec_app_fico_range_high,float64
118,sec_app_earliest_cr_line,object
119,sec_app_inq_last_6mths,float64
120,sec_app_mort_acc,float64
121,sec_app_open_acc,float64
122,sec_app_revol_util,float64
123,sec_app_open_act_il,float64
124,sec_app_num_rev_accts,float64


## 5.2 Secondary-Applicant Credit Feature Audit

This section evaluates variables describing the secondary applicant in joint loan applications.

These variables generally represent legitimate application-time or credit-bureau information. However, they are relevant only when the application type is joint. Their final inclusion in the modelling dataset will therefore depend on joint-application prevalence, missingness, and preprocessing strategy.

In [36]:
features_115_127_audit = {

    "revol_bal_joint": {
        "definition": "Combined revolving credit balance for joint applicants.",
        "information_timing": "Application / joint credit bureau",
        "decision": "Keep",
        "justification": "Available for joint applications before loan performance is observed and relevant to combined indebtedness."
    },

    "sec_app_fico_range_low": {
        "definition": "Lower boundary of the secondary applicant's FICO score range.",
        "information_timing": "Application / secondary-applicant credit bureau",
        "decision": "Keep",
        "justification": "Secondary-applicant FICO information is available during underwriting for joint applications."
    },

    "sec_app_fico_range_high": {
        "definition": "Upper boundary of the secondary applicant's FICO score range.",
        "information_timing": "Application / secondary-applicant credit bureau",
        "decision": "Keep",
        "justification": "Secondary-applicant FICO information is available during underwriting for joint applications."
    },

    "sec_app_earliest_cr_line": {
        "definition": "Date of the secondary applicant's earliest reported credit line.",
        "information_timing": "Application / secondary-applicant credit bureau",
        "decision": "Keep",
        "justification": "Credit-history age of the secondary applicant is observable before loan origination."
    },

    "sec_app_inq_last_6mths": {
        "definition": "Number of credit enquiries for the secondary applicant during the previous six months.",
        "information_timing": "Application / secondary-applicant credit bureau",
        "decision": "Keep",
        "justification": "Recent enquiry activity is available during underwriting."
    },

    "sec_app_mort_acc": {
        "definition": "Number of mortgage accounts belonging to the secondary applicant.",
        "information_timing": "Application / secondary-applicant credit bureau",
        "decision": "Keep",
        "justification": "Mortgage-account history is observable during underwriting."
    },

    "sec_app_open_acc": {
        "definition": "Number of open accounts belonging to the secondary applicant.",
        "information_timing": "Application / secondary-applicant credit bureau",
        "decision": "Keep",
        "justification": "Open-account information is available at underwriting."
    },

    "sec_app_revol_util": {
        "definition": "Revolving credit utilisation of the secondary applicant.",
        "information_timing": "Application / secondary-applicant credit bureau",
        "decision": "Keep",
        "justification": "Existing utilisation is observable before loan performance."
    },

    "sec_app_open_act_il": {
        "definition": "Number of active installment accounts belonging to the secondary applicant.",
        "information_timing": "Application / secondary-applicant credit bureau",
        "decision": "Keep",
        "justification": "Current installment-account activity is available during underwriting."
    },

    "sec_app_num_rev_accts": {
        "definition": "Number of revolving accounts belonging to the secondary applicant.",
        "information_timing": "Application / secondary-applicant credit bureau",
        "decision": "Keep",
        "justification": "Revolving-account history is observable before origination."
    },

    "sec_app_chargeoff_within_12_mths": {
        "definition": "Number of secondary-applicant charge-offs during the previous 12 months.",
        "information_timing": "Application / secondary-applicant credit bureau",
        "decision": "Keep",
        "justification": "Represents historical adverse credit behaviour prior to the new loan."
    },

    "sec_app_collections_12_mths_ex_med": {
        "definition": "Number of secondary-applicant collections during the previous 12 months, excluding medical collections.",
        "information_timing": "Application / secondary-applicant credit bureau",
        "decision": "Keep",
        "justification": "Historical collection activity is available during underwriting."
    },

    "sec_app_mths_since_last_major_derog": {
        "definition": "Number of months since the secondary applicant's most recent major derogatory credit event.",
        "information_timing": "Application / secondary-applicant credit bureau",
        "decision": "Keep",
        "justification": "Historical derogatory-credit information is available before loan origination."
    }
}

In [37]:
for feature, audit in features_115_127_audit.items():

    mask = feature_inventory["feature"] == feature

    feature_inventory.loc[mask, "definition"] = audit["definition"]
    feature_inventory.loc[mask, "information_timing"] = audit["information_timing"]
    feature_inventory.loc[mask, "decision"] = audit["decision"]
    feature_inventory.loc[mask, "justification"] = audit["justification"]

In [38]:
feature_inventory.iloc[115:128]

,feature,dtype,definition,information_timing,decision,justification
115,revol_bal_joint,float64,Combined revolving credit balance for joint ap...,Application / joint credit bureau,Keep,Available for joint applications before loan p...
116,sec_app_fico_range_low,float64,Lower boundary of the secondary applicant's FI...,Application / secondary-applicant credit bureau,Keep,Secondary-applicant FICO information is availa...
117,sec_app_fico_range_high,float64,Upper boundary of the secondary applicant's FI...,Application / secondary-applicant credit bureau,Keep,Secondary-applicant FICO information is availa...
118,sec_app_earliest_cr_line,object,Date of the secondary applicant's earliest rep...,Application / secondary-applicant credit bureau,Keep,Credit-history age of the secondary applicant ...
119,sec_app_inq_last_6mths,float64,Number of credit enquiries for the secondary a...,Application / secondary-applicant credit bureau,Keep,Recent enquiry activity is available during un...
120,sec_app_mort_acc,float64,Number of mortgage accounts belonging to the s...,Application / secondary-applicant credit bureau,Keep,Mortgage-account history is observable during ...
121,sec_app_open_acc,float64,Number of open accounts belonging to the secon...,Application / secondary-applicant credit bureau,Keep,Open-account information is available at under...
122,sec_app_revol_util,float64,Revolving credit utilisation of the secondary ...,Application / secondary-applicant credit bureau,Keep,Existing utilisation is observable before loan...
123,sec_app_open_act_il,float64,Number of active installment accounts belongin...,Application / secondary-applicant credit bureau,Keep,Current installment-account activity is availa...
124,sec_app_num_rev_accts,float64,Number of revolving accounts belonging to the ...,Application / secondary-applicant credit bureau,Keep,Revolving-account history is observable before...


In [39]:
audited = feature_inventory["decision"].ne("").sum()
remaining = feature_inventory["decision"].eq("").sum()

print("Total features :", len(feature_inventory))
print("Audited        :", audited)
print("Remaining      :", remaining)

Total features : 151
Audited        : 128
Remaining      : 23


# 6. Hardship-Related Target Leakage Analysis

## Purpose

This section evaluates variables associated with LendingClub hardship arrangements.

Hardship information describes borrower financial distress and loan-servicing interventions that occur after loan origination. Because the proposed credit-risk model is intended to estimate default risk using information available at or before the lending decision, hardship-related variables are not legitimate predictors.

Their inclusion could reveal subsequent borrower distress or repayment difficulty and would therefore introduce target leakage.

In [40]:
features_128_142_audit = {

    "hardship_flag": {
        "definition": "Indicates whether the borrower is currently enrolled in a hardship programme.",
        "information_timing": "Post-origination / servicing",
        "decision": "Remove",
        "justification": "Hardship enrolment occurs after origination and directly signals subsequent borrower financial distress."
    },

    "hardship_type": {
        "definition": "Type of hardship programme associated with the loan.",
        "information_timing": "Post-origination / servicing",
        "decision": "Remove",
        "justification": "Hardship programme information is generated after origination and may reveal repayment difficulty."
    },

    "hardship_reason": {
        "definition": "Reason recorded for the borrower's hardship arrangement.",
        "information_timing": "Post-origination / servicing",
        "decision": "Remove",
        "justification": "The reason for hardship is observed after origination and contains future information about borrower distress."
    },

    "hardship_status": {
        "definition": "Status of the hardship arrangement.",
        "information_timing": "Post-origination / servicing",
        "decision": "Remove",
        "justification": "Hardship status reflects post-origination servicing activity."
    },

    "deferral_term": {
        "definition": "Length of the payment deferral associated with the hardship arrangement.",
        "information_timing": "Post-origination / servicing",
        "decision": "Remove",
        "justification": "Payment deferral terms arise after a hardship event and are unavailable at the original lending decision."
    },

    "hardship_amount": {
        "definition": "Payment amount associated with the hardship arrangement.",
        "information_timing": "Post-origination / servicing",
        "decision": "Remove",
        "justification": "The hardship payment amount is determined after origination and reflects subsequent servicing activity."
    },

    "hardship_start_date": {
        "definition": "Start date of the hardship arrangement.",
        "information_timing": "Post-origination / servicing",
        "decision": "Remove",
        "justification": "The hardship start date occurs after loan origination and therefore represents future information."
    },

    "hardship_end_date": {
        "definition": "End date of the hardship arrangement.",
        "information_timing": "Post-origination / servicing",
        "decision": "Remove",
        "justification": "The hardship end date is generated after origination and cannot be known at application time."
    },

    "payment_plan_start_date": {
        "definition": "Date on which the hardship-related payment plan begins.",
        "information_timing": "Post-origination / servicing",
        "decision": "Remove",
        "justification": "This date reflects a subsequent loan-servicing intervention."
    },

    "hardship_length": {
        "definition": "Length of the hardship arrangement.",
        "information_timing": "Post-origination / servicing",
        "decision": "Remove",
        "justification": "Hardship duration is determined only after the loan has entered a hardship programme."
    },

    "hardship_dpd": {
        "definition": "Days past due associated with the hardship record.",
        "information_timing": "Post-origination / delinquency",
        "decision": "Remove",
        "justification": "Days-past-due information directly reflects subsequent repayment difficulty and therefore constitutes target leakage."
    },

    "hardship_loan_status": {
        "definition": "Loan status associated with the hardship period.",
        "information_timing": "Post-origination / loan outcome",
        "decision": "Remove",
        "justification": "The variable contains post-origination loan-performance information closely related to the prediction target."
    },

    "orig_projected_additional_accrued_interest": {
        "definition": "Projected additional interest expected to accrue under the hardship arrangement.",
        "information_timing": "Post-origination / servicing",
        "decision": "Remove",
        "justification": "This value is calculated only after a hardship arrangement exists and therefore contains future servicing information."
    },

    "hardship_payoff_balance_amount": {
        "definition": "Loan payoff balance associated with the hardship arrangement.",
        "information_timing": "Post-origination / servicing",
        "decision": "Remove",
        "justification": "The payoff balance is determined after origination and reflects subsequent loan performance."
    },

    "hardship_last_payment_amount": {
        "definition": "Most recent payment amount associated with the hardship record.",
        "information_timing": "Post-origination / repayment",
        "decision": "Remove",
        "justification": "The variable records realised payment behaviour occurring after origination."
    }
}

In [41]:
for feature, audit in features_128_142_audit.items():

    mask = feature_inventory["feature"] == feature

    feature_inventory.loc[mask, "definition"] = audit["definition"]
    feature_inventory.loc[mask, "information_timing"] = audit["information_timing"]
    feature_inventory.loc[mask, "decision"] = audit["decision"]
    feature_inventory.loc[mask, "justification"] = audit["justification"]

In [42]:
feature_inventory.iloc[128:143]

,feature,dtype,definition,information_timing,decision,justification
128,hardship_flag,object,Indicates whether the borrower is currently en...,Post-origination / servicing,Remove,Hardship enrolment occurs after origination an...
129,hardship_type,object,Type of hardship programme associated with the...,Post-origination / servicing,Remove,Hardship programme information is generated af...
130,hardship_reason,object,Reason recorded for the borrower's hardship ar...,Post-origination / servicing,Remove,The reason for hardship is observed after orig...
131,hardship_status,object,Status of the hardship arrangement.,Post-origination / servicing,Remove,Hardship status reflects post-origination serv...
132,deferral_term,float64,Length of the payment deferral associated with...,Post-origination / servicing,Remove,Payment deferral terms arise after a hardship ...
133,hardship_amount,float64,Payment amount associated with the hardship ar...,Post-origination / servicing,Remove,The hardship payment amount is determined afte...
134,hardship_start_date,object,Start date of the hardship arrangement.,Post-origination / servicing,Remove,The hardship start date occurs after loan orig...
135,hardship_end_date,object,End date of the hardship arrangement.,Post-origination / servicing,Remove,The hardship end date is generated after origi...
136,payment_plan_start_date,object,Date on which the hardship-related payment pla...,Post-origination / servicing,Remove,This date reflects a subsequent loan-servicing...
137,hardship_length,float64,Length of the hardship arrangement.,Post-origination / servicing,Remove,Hardship duration is determined only after the...


In [43]:
audited = feature_inventory["decision"].ne("").sum()
remaining = feature_inventory["decision"].eq("").sum()

print("Total features :", len(feature_inventory))
print("Audited        :", audited)
print("Remaining      :", remaining)

Total features : 151
Audited        : 143
Remaining      : 8


## Discussion

All hardship-related variables were classified as post-origination information.

These variables describe hardship enrolment, payment deferral, delinquency, revised payment arrangements, accrued interest, payoff balances, and payments occurring after the loan has been issued.

Although these variables may be highly predictive of charge-off, that predictive power would be inappropriate for the proposed application-time credit-risk model because the information becomes available only after borrower repayment behaviour has begun.

## Key Findings

- All 15 hardship-related variables were classified for removal.
- Hardship variables provide direct evidence of subsequent financial distress.
- `hardship_dpd` and `hardship_loan_status` are particularly strong examples of target leakage.
- High predictive power from these variables would not represent genuine prospective credit-risk prediction.

## Research Implication

Hardship-related variables will be excluded from all modelling experiments to ensure that structured-only and hybrid models operate under a consistent application-time information constraint.

# 7. Disbursement and Debt-Settlement Feature Audit

## Purpose

The final feature-audit section examines the loan disbursement method and variables associated with debt-settlement arrangements.

Debt-settlement information is generated after loan origination when repayment difficulties have already occurred. Such variables may contain strong information about subsequent loan performance and therefore require exclusion from an application-time credit-risk model.

The disbursement method is considered separately because it is associated with loan origination rather than subsequent repayment behaviour.

In [44]:
features_143_150_audit = {

    "disbursement_method": {
        "definition": "Method by which the loan proceeds were disbursed.",
        "information_timing": "Origination / loan administration",
        "decision": "Review",
        "justification": "The variable is associated with loan origination rather than future repayment behaviour, but it is an administrative characteristic rather than a direct borrower-risk attribute. Its suitability should therefore be reviewed before modelling."
    },

    "debt_settlement_flag": {
        "definition": "Indicates whether the borrower has entered into a debt-settlement arrangement.",
        "information_timing": "Post-origination / debt settlement",
        "decision": "Remove",
        "justification": "Debt-settlement participation occurs after repayment difficulty has emerged and therefore contains future outcome information."
    },

    "debt_settlement_flag_date": {
        "definition": "Date associated with the debt-settlement flag.",
        "information_timing": "Post-origination / debt settlement",
        "decision": "Remove",
        "justification": "The date is generated after origination when a debt-settlement arrangement is established."
    },

    "settlement_status": {
        "definition": "Status of the borrower's debt-settlement arrangement.",
        "information_timing": "Post-origination / debt settlement",
        "decision": "Remove",
        "justification": "Settlement status reflects subsequent servicing and borrower repayment difficulty."
    },

    "settlement_date": {
        "definition": "Date on which the debt-settlement arrangement was established.",
        "information_timing": "Post-origination / debt settlement",
        "decision": "Remove",
        "justification": "The settlement date occurs after loan origination and therefore cannot be available at the original lending decision."
    },

    "settlement_amount": {
        "definition": "Amount associated with the debt-settlement arrangement.",
        "information_timing": "Post-origination / debt settlement",
        "decision": "Remove",
        "justification": "The settlement amount is determined after repayment difficulties arise and directly reflects subsequent loan servicing."
    },

    "settlement_percentage": {
        "definition": "Settlement amount expressed relative to the outstanding loan balance.",
        "information_timing": "Post-origination / debt settlement",
        "decision": "Remove",
        "justification": "This variable is calculated from a post-origination settlement arrangement and therefore contains future information."
    },

    "settlement_term": {
        "definition": "Number of months associated with the debt-settlement arrangement.",
        "information_timing": "Post-origination / debt settlement",
        "decision": "Remove",
        "justification": "Settlement terms are established only after repayment difficulties have occurred."
    }
}

In [45]:
for feature, audit in features_143_150_audit.items():

    mask = feature_inventory["feature"] == feature

    feature_inventory.loc[mask, "definition"] = audit["definition"]
    feature_inventory.loc[mask, "information_timing"] = audit["information_timing"]
    feature_inventory.loc[mask, "decision"] = audit["decision"]
    feature_inventory.loc[mask, "justification"] = audit["justification"]

In [46]:
feature_inventory.iloc[143:151]

,feature,dtype,definition,information_timing,decision,justification
143,disbursement_method,object,Method by which the loan proceeds were disbursed.,Origination / loan administration,Review,The variable is associated with loan originati...
144,debt_settlement_flag,object,Indicates whether the borrower has entered int...,Post-origination / debt settlement,Remove,Debt-settlement participation occurs after rep...
145,debt_settlement_flag_date,object,Date associated with the debt-settlement flag.,Post-origination / debt settlement,Remove,The date is generated after origination when a...
146,settlement_status,object,Status of the borrower's debt-settlement arran...,Post-origination / debt settlement,Remove,Settlement status reflects subsequent servicin...
147,settlement_date,object,Date on which the debt-settlement arrangement ...,Post-origination / debt settlement,Remove,The settlement date occurs after loan originat...
148,settlement_amount,float64,Amount associated with the debt-settlement arr...,Post-origination / debt settlement,Remove,The settlement amount is determined after repa...
149,settlement_percentage,float64,Settlement amount expressed relative to the ou...,Post-origination / debt settlement,Remove,This variable is calculated from a post-origin...
150,settlement_term,float64,Number of months associated with the debt-sett...,Post-origination / debt settlement,Remove,Settlement terms are established only after re...


In [47]:
audited = feature_inventory["decision"].ne("").sum()
remaining = feature_inventory["decision"].eq("").sum()

print("Total features :", len(feature_inventory))
print("Audited        :", audited)
print("Remaining      :", remaining)

Total features : 151
Audited        : 151
Remaining      : 0


In [48]:
feature_inventory["decision"].value_counts()

,count
decision,
Keep,95
Remove,40
Review,15
Target,1


In [49]:
# Let's analyse the features for which we've marked decision as 'Review'
review_features = feature_inventory[
    feature_inventory["decision"] == "Review"
][
    ["feature", "definition", "information_timing", "justification"]
]

review_features

,feature,definition,information_timing,justification
3,funded_amnt,The total amount committed to that loan at tha...,Funding / origination outcome,Represents the amount ultimately committed rat...
4,funded_amnt_inv,The total amount committed by investors for th...,Funding / origination outcome,Reflects investor funding activity and may not...
6,int_rate,Interest rate on the loan.,Lender underwriting / origination,Available around origination but may already e...
7,installment,The monthly payment owed by the borrower if th...,Lender underwriting / origination,Known before repayment begins but is derived f...
8,grade,LC assigned loan grade.,Lender underwriting / origination,"Not future-outcome leakage, but it directly in..."
9,sub_grade,LC assigned loan subgrade.,Lender underwriting / origination,Provides a more granular LendingClub risk asse...
10,emp_title,The job title supplied by the borrower when ap...,Application / borrower-provided,"Available at application time, but the variabl..."
15,issue_d,The month in which the loan was funded.,Origination / temporal metadata,The funding month should not automatically be ...
17,pymnt_plan,Indicates whether a payment plan has been put ...,Potentially post-origination / servicing,The timing and meaning of the payment-plan ind...
21,title,The loan title provided by the borrower.,Application / borrower-provided text,"Available at application, but it may overlap s..."


In [50]:
print(review_features["feature"].tolist())

['funded_amnt', 'funded_amnt_inv', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'issue_d', 'pymnt_plan', 'title', 'zip_code', 'addr_state', 'initial_list_status', 'policy_code', 'disbursement_method']


In [51]:
# Let's diagnoize the 15 review features to confirm whether to keep or remove
review_cols = df.columns[df.columns.isin(review_features["feature"])]

review_diagnostics = []

for col in review_cols:
    review_diagnostics.append({
        "feature": col,
        "dtype": str(df[col].dtype),
        "missing_count": df[col].isna().sum(),
        "missing_pct": round(df[col].isna().mean() * 100, 2),
        "unique_values": df[col].nunique(dropna=True),
        "sample_values": df[col].dropna().astype(str).unique()[:5].tolist()
    })

review_diagnostics = pd.DataFrame(review_diagnostics)

review_diagnostics

,feature,dtype,missing_count,missing_pct,unique_values,sample_values
0,funded_amnt,float64,33,0.00,1572,"[3600.0, 24700.0, 20000.0, 35000.0, 10400.0]"
1,funded_amnt_inv,float64,33,0.00,10057,"[3600.0, 24700.0, 20000.0, 35000.0, 10400.0]"
2,int_rate,float64,33,0.00,673,"[13.99, 11.99, 10.78, 14.85, 22.45]"
3,installment,float64,33,0.00,93301,"[123.03, 820.28, 432.66, 829.9, 289.91]"
4,grade,object,33,0.00,7,"[C, B, F, A, E]"
5,sub_grade,object,33,0.00,35,"[C4, C1, B4, C5, F1]"
6,emp_title,object,167002,7.39,512694,"[leadman, Engineer, truck driver, Information ..."
7,issue_d,object,33,0.00,139,"[Dec-2015, Nov-2015, Oct-2015, Sep-2015, Aug-2..."
8,pymnt_plan,object,33,0.00,2,"[n, y]"
9,title,object,23359,1.03,63154,"[Debt consolidation, Business, Major purchase,..."


### Resolution of Features Initially Marked for Review

The 15 variables initially marked `Review` were assessed using their information
timing, role in the lending process, missingness, cardinality, and overlap with
other retained predictors.

They are excluded from the primary modelling feature set for one or more of the
following reasons:

- **Lender-generated underwriting or pricing signals** (`int_rate`, `grade`,
  `sub_grade`, `installment`) may encode LendingClub's existing assessment
  rather than independent borrower information.
- **Funding/origination outcome variables** (`funded_amnt`,
  `funded_amnt_inv`) reflect realised funding rather than the original borrower
  request.
- **High-cardinality or weakly generalisable fields** (`emp_title`, `zip_code`,
  `addr_state`) may introduce sparse, location-specific, or occupational proxy
  effects without being necessary to answer the research questions.
- **Redundant or overlapping text/administrative variables** (`title`,
  `pymnt_plan`, `initial_list_status`, `policy_code`,
  `disbursement_method`) add limited independent application-time signal for
  the present research design.
- **`issue_d`** is not used as a predictive feature; it is retained separately
  only to construct chronological train, validation, and test cohorts.

This exclusion is a modelling-scope decision rather than a claim that these
variables are intrinsically unusable in all credit-risk applications.


In [52]:
# Apply the documented audit decision to the 15 reviewed features
feature_inventory.loc[
    feature_inventory["feature"].isin(review_cols),
    "decision"
] = "Remove"

# Print the decision status counts
feature_inventory["decision"].value_counts()


,count
decision,
Keep,95
Remove,55
Target,1


# 8. Initial Feature Audit Summary

A systematic audit was performed on all 151 variables in the original LendingClub dataset to determine their suitability for application-time credit-risk prediction.

Each feature was assessed according to its information timing, relationship to the lending process, potential for target leakage, and relevance to the proposed modelling task. Features initially requiring further investigation were additionally examined using their missingness, cardinality, value distribution, and conceptual role.

### Final Audit Outcome

- **95 features — Keep:** Variables considered eligible for further preprocessing and feature-selection analysis.
- **55 features — Remove:** Variables excluded because they represent identifiers, post-origination information, repayment or recovery behaviour, hardship or settlement information, LendingClub-generated underwriting signals, administrative metadata, redundant/high-cardinality information, or otherwise unsuitable predictors.
- **1 feature — Target:** `loan_status`, which will be used to construct the binary prediction target.

### Target Leakage Control

Particular attention was given to preventing target leakage. Variables describing realised payments, outstanding principal, recoveries, updated FICO information, hardship arrangements, and debt settlements were excluded because they contain information generated after loan origination.

LendingClub-generated underwriting variables such as `grade`, `sub_grade`, and `int_rate` were also excluded from the primary predictor set. Although available around origination, these variables encode aspects of the platform's existing underwriting and risk-assessment process. Their exclusion supports a more independent evaluation of predictive information derived from borrower and credit characteristics.

### Text Feature

The borrower-written `desc` variable was retained because it represents application-time unstructured information and is central to the proposed comparison of textual representations and hybrid structured-text credit-risk models.

### Key Finding

The feature audit reduced the original 151-variable dataset to **95 candidate structured/text predictors and one target variable**.

These 95 variables should not yet be interpreted as the final modelling feature set. They have passed the initial suitability and leakage audit, but will subsequently be evaluated for missingness, low variance, redundancy, cardinality, transformation requirements, and other preprocessing considerations before the final predictor matrix is constructed.

# 9. Data Quality Assessment of Candidate Features

## 9.1 Missing-Value Analysis

Following the feature eligibility and target-leakage audit, 95 variables remain as candidate predictors.

The next stage evaluates the data quality of these candidate variables. Missingness is examined before preprocessing because variables with very high proportions of missing observations may provide limited usable information and may require exclusion rather than imputation.

At this stage, no variables are removed automatically. The missing-value distribution is first quantified and inspected before appropriate thresholds and treatment strategies are determined.

In [53]:
candidate_features = feature_inventory.loc[
    feature_inventory["decision"] == "Keep",
    "feature"
].tolist()

print("Number of candidate features:", len(candidate_features))

Number of candidate features: 95


In [54]:
def analyze_missingness(df, features):
    """
    Calculate feature-level missingness and summarize it into percentage bands.
    """

    missing_analysis = pd.DataFrame({
        "feature": features,
        "missing_count": [df[col].isna().sum() for col in features],
        "missing_pct": [df[col].isna().mean() * 100 for col in features]
    })

    missing_analysis["missing_pct"] = (
        missing_analysis["missing_pct"].round(2)
    )

    missing_analysis = (
        missing_analysis
        .sort_values("missing_pct", ascending=False)
        .reset_index(drop=True)
    )

    missing_summary = pd.Series({
        "0% missing":
            (missing_analysis["missing_pct"] == 0).sum(),

        ">0% to 10%":
            ((missing_analysis["missing_pct"] > 0) &
             (missing_analysis["missing_pct"] <= 10)).sum(),

        ">10% to 30%":
            ((missing_analysis["missing_pct"] > 10) &
             (missing_analysis["missing_pct"] <= 30)).sum(),

        ">30% to 50%":
            ((missing_analysis["missing_pct"] > 30) &
             (missing_analysis["missing_pct"] <= 50)).sum(),

        ">50%":
            (missing_analysis["missing_pct"] > 50).sum()
    })

    return missing_analysis, missing_summary

In [55]:
missing_analysis, missing_summary = analyze_missingness(
    df,
    candidate_features
)

print(missing_analysis)
print(missing_summary)


                                feature  missing_count  missing_pct
0   sec_app_mths_since_last_major_derog        2224759        98.41
1                    sec_app_revol_util        2154517        95.30
2      sec_app_chargeoff_within_12_mths        2152680        95.22
3                      sec_app_mort_acc        2152680        95.22
4                       revol_bal_joint        2152681        95.22
5                 sec_app_num_rev_accts        2152680        95.22
6              sec_app_earliest_cr_line        2152680        95.22
7                   sec_app_open_act_il        2152680        95.22
8    sec_app_collections_12_mths_ex_med        2152680        95.22
9                sec_app_fico_range_low        2152680        95.22
10               sec_app_inq_last_6mths        2152680        95.22
11                     sec_app_open_acc        2152680        95.22
12              sec_app_fico_range_high        2152680        95.22
13            verification_status_joint        2

In [56]:
missing_analysis[
    missing_analysis["missing_pct"] > 50
]

,feature,missing_count,missing_pct
0,sec_app_mths_since_last_major_derog,2224759,98.41
1,sec_app_revol_util,2154517,95.30
2,sec_app_chargeoff_within_12_mths,2152680,95.22
3,sec_app_mort_acc,2152680,95.22
4,revol_bal_joint,2152681,95.22
5,sec_app_num_rev_accts,2152680,95.22
6,sec_app_earliest_cr_line,2152680,95.22
7,sec_app_open_act_il,2152680,95.22
8,sec_app_collections_12_mths_ex_med,2152680,95.22
9,sec_app_fico_range_low,2152680,95.22


In [57]:
desc_stats = {
    "total_rows": len(df),
    "desc_present": df["desc"].notna().sum(),
    "desc_missing": df["desc"].isna().sum(),
    "desc_present_pct": round(df["desc"].notna().mean() * 100, 2)
}

desc_stats

{'total_rows': 2260701,
 'desc_present': np.int64(126065),
 'desc_missing': np.int64(2134636),
 'desc_present_pct': np.float64(5.58)}

### Findings: Treatment of the `desc` Variable

The `desc` variable exhibits approximately 94.42% missingness in the complete LendingClub dataset. However, `desc` is retained because borrower-written text is a core input required for the textual and hybrid modelling experiments defined in this research.

The availability and characteristics of `desc`, including the usable text population and associated target distribution, were examined previously during dataset understanding and exploratory analysis in Notebook 1. These analyses are therefore not repeated here.

Consequently, `desc` is treated separately from the structured-feature missingness filtering process. The textual and hybrid experiments will operate on the eligible subset of observations containing usable borrower descriptions.

## 9.2 Structural Missingness in Joint-Application Features (Count: 16)

Several joint and secondary-applicant variables exhibit approximately 95% missingness. These variables are only applicable to joint loan applications and therefore require separate investigation before applying a general missing-value threshold.

The relationship between `application_type` and the availability of joint-applicant variables is examined below to distinguish structural non-applicability from ordinary missing data.

In [58]:
df["application_type"].value_counts(dropna=False)

,count
application_type,
Individual,2139958
Joint App,120710
NaN,33


In [59]:
df["application_type"].value_counts(
    normalize=True,
    dropna=False
).mul(100).round(2)

# ~95% applications are individual applicants where as ~5% are joint.

,proportion
application_type,
Individual,94.66
Joint App,5.34
NaN,0.00


In [60]:
joint_check = df.groupby("application_type")[
    [
        "annual_inc_joint",
        "dti_joint",
        "sec_app_fico_range_low"
    ]
].agg(
    lambda x: round(x.notna().mean() * 100, 2)
)

joint_check

# Note: From the result clear that those features are not populated for indvidual applicant type.
# Concludes that those feature names starting ending with '_joint' or starting with 'sec_' are only applicable for joint applications.

,annual_inc_joint,dti_joint,sec_app_fico_range_low
application_type,,,
Individual,0.0,0.0,0.00
Joint App,100.0,100.0,89.49


In [61]:
joint_features_to_remove = [
    "annual_inc_joint",
    "dti_joint",
    "verification_status_joint",
    "revol_bal_joint",
    "sec_app_fico_range_low",
    "sec_app_fico_range_high",
    "sec_app_earliest_cr_line",
    "sec_app_inq_last_6mths",
    "sec_app_mort_acc",
    "sec_app_open_acc",
    "sec_app_revol_util",
    "sec_app_open_act_il",
    "sec_app_num_rev_accts",
    "sec_app_chargeoff_within_12_mths",
    "sec_app_collections_12_mths_ex_med",
    "sec_app_mths_since_last_major_derog"
]

In [62]:
feature_inventory.loc[
    feature_inventory["feature"].isin(joint_features_to_remove),
    "decision"
] = "Remove"

In [63]:
feature_inventory["decision"].value_counts()

,count
decision,
Keep,79
Remove,71
Target,1


### Joint-Application Feature Decision

Joint applications account for only 5.34% of the complete dataset, while 94.66% of applications are individual applications.

The high missingness observed in joint and secondary-applicant variables was therefore identified as predominantly structural rather than ordinary missingness. Representative joint-income and debt-to-income variables were populated for 100% of joint applications but 0% of individual applications, confirming that these variables are conditionally applicable.

Although these variables contain legitimate application-time information for joint applications, they were excluded from the primary modelling feature set because they are non-applicable to the large majority of observations and would require a separate conditional preprocessing strategy for a relatively small subgroup.

Joint-application observations themselves are retained, and `application_type` remains available as a predictor. This preserves information about whether an application is individual or joint while maintaining a consistent structured feature space across observations.

## 9.3 Missingness in Delinquency and Derogatory-History Features (count 5)

Several credit-history variables exhibit substantial missingness despite representing potentially relevant borrower-risk information.

Unlike the joint-applicant variables examined previously, these features describe the elapsed time since specific adverse credit events. Their missingness may therefore be informative; for example, an unavailable "months since last delinquency" value may be associated with borrowers who have no recorded delinquency event.

Before deciding whether these variables should be removed or imputed, their missingness is examined in relation to other credit-history indicators.

In [64]:
delinq_missing_features = [
    "mths_since_last_record",
    "mths_since_recent_bc_dlq",
    "mths_since_last_major_derog",
    "mths_since_recent_revol_delinq",
    "mths_since_last_delinq"
]

In [65]:
df[delinq_missing_features].isna().mean().mul(100).round(2)

,0
mths_since_last_record,84.11
mths_since_recent_bc_dlq,77.01
mths_since_last_major_derog,74.31
mths_since_recent_revol_delinq,67.25
mths_since_last_delinq,51.25


In [66]:
df["mths_since_last_delinq_missing"] = (
    df["mths_since_last_delinq"].isna()
)

pd.crosstab(
    df["mths_since_last_delinq_missing"],
    df["delinq_2yrs"],
    normalize="index"
).iloc[:, :10].round(4)

delinq_2yrs,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0
mths_since_last_delinq_missing,,,,,,,,,,
False,0.6219,0.2522,0.0732,0.0266,0.0118,0.0059,0.0033,0.0018,0.0011,0.0007
True,0.9959,0.0029,0.0006,0.0002,0.0001,0.0001,0.0001,0.0000,0.0000,0.0000


In [67]:
df.groupby(
    "mths_since_last_delinq_missing"
)["delinq_2yrs"].agg(
    ["count", "mean", "median", "min", "max"]
)

,count,mean,median,min,max
mths_since_last_delinq_missing,,,,,
False,1102166,0.621765,0.0,0.0,58.0
True,1158473,0.007298,0.0,0.0,26.0


In [68]:
# Let's explicitly compare whether borrowers have any delinquency in the last two years:

df["has_recent_delinq"] = df["delinq_2yrs"] > 0

pd.crosstab(
    df["mths_since_last_delinq_missing"],
    df["has_recent_delinq"],
    normalize="index"
).mul(100).round(2)

has_recent_delinq,False,True
mths_since_last_delinq_missing,,
False,62.19,37.81
True,99.59,0.41


### Finding: `mths_since_last_delinq`

Missingness in `mths_since_last_delinq` appears to be strongly associated with the absence of recent delinquency activity. Among observations where `mths_since_last_delinq` is missing, 99.59% have no recorded delinquency within the previous two years (`delinq_2yrs = 0`).

This indicates that the missingness is potentially informative rather than simply representing random data loss. However, `delinq_2yrs` only represents delinquency within a limited historical window; therefore, a missing value cannot be interpreted conclusively as evidence that the borrower has never experienced a delinquency.

The feature is consequently retained for modelling, with its missingness to be explicitly preserved during preprocessing rather than treated using simple unconditional imputation.

In [69]:
remaining_history_features = [
    "mths_since_last_record",
    "mths_since_recent_bc_dlq",
    "mths_since_last_major_derog",
    "mths_since_recent_revol_delinq"
]

history_diagnostics = pd.DataFrame({
    "feature": remaining_history_features,
    "missing_pct": [
        round(df[col].isna().mean() * 100, 2)
        for col in remaining_history_features
    ],
    "non_missing_count": [
        df[col].notna().sum()
        for col in remaining_history_features
    ],
    "unique_values": [
        df[col].nunique(dropna=True)
        for col in remaining_history_features
    ],
    "median_when_present": [
        df[col].median()
        for col in remaining_history_features
    ]
})

history_diagnostics

,feature,missing_pct,non_missing_count,unique_values,median_when_present
0,mths_since_last_record,84.11,359156,129,74.0
1,mths_since_recent_bc_dlq,77.01,519701,177,37.0
2,mths_since_last_major_derog,74.31,580775,183,44.0
3,mths_since_recent_revol_delinq,67.25,740359,179,33.0


In [70]:
resolved_df = df[
    df["loan_status"].isin(["Fully Paid", "Charged Off"])
].copy()

resolved_df["default_flag"] = (
    resolved_df["loan_status"] == "Charged Off"
).astype(int)

In [71]:
history_missingness_results = []

for col in delinq_missing_features:

    temp = resolved_df.groupby(
        resolved_df[col].isna()
    )["default_flag"].agg(
        count="count",
        default_rate="mean"
    )

    for is_missing, row in temp.iterrows():
        history_missingness_results.append({
            "feature": col,
            "is_missing": is_missing,
            "count": int(row["count"]),
            "default_rate_pct": round(row["default_rate"] * 100, 2)
        })

history_missingness_results = pd.DataFrame(
    history_missingness_results
)

history_missingness_results

,feature,is_missing,count,default_rate_pct
0,mths_since_last_record,False,228555,22.74
1,mths_since_last_record,True,1116755,19.39
2,mths_since_recent_bc_dlq,False,319020,20.98
3,mths_since_recent_bc_dlq,True,1026290,19.65
4,mths_since_last_major_derog,False,353750,21.94
5,mths_since_last_major_derog,True,991560,19.26
6,mths_since_recent_revol_delinq,False,449962,20.68
7,mths_since_recent_revol_delinq,True,895348,19.60
8,mths_since_last_delinq,False,666567,20.68
9,mths_since_last_delinq,True,678743,19.26


### Decision: Delinquency and Derogatory-History Features

The five high-missingness credit-history variables were retained despite missingness ranging from 51.25% to 84.11%.

The investigation indicated that their missingness is potentially informative rather than representing ordinary random data loss. For `mths_since_last_delinq`, 99.59% of observations with a missing value had no recorded delinquency within the previous two years. Furthermore, across all five variables, observations with recorded values exhibited somewhat higher default rates than observations with missing values.

These findings are consistent with the interpretation that the availability of these variables is associated with the occurrence of relevant adverse credit events. However, missing values are not interpreted conclusively as evidence that an adverse event has never occurred, because individual variables may cover different historical windows and credit-event definitions.

Accordingly, all five variables are retained as candidate predictors. During model preprocessing, missingness will be explicitly preserved, for example through missing-value indicators combined with numerical imputation, rather than relying on unconditional imputation alone.

In [72]:
feature_inventory["decision"].value_counts()

,count
decision,
Keep,79
Remove,71
Target,1


In [73]:
candidate_features = feature_inventory.loc[
    feature_inventory["decision"] == "Keep",
    "feature"
].tolist()

missing_analysis, missing_summary = analyze_missingness(
    df,
    candidate_features
)

print("Current candidate features:", len(candidate_features))
print(missing_analysis)
print(missing_summary)

Current candidate features: 79
                           feature  missing_count  missing_pct
0                             desc        2134636        94.42
1           mths_since_last_record        1901545        84.11
2         mths_since_recent_bc_dlq        1741000        77.01
3      mths_since_last_major_derog        1679926        74.31
4   mths_since_recent_revol_delinq        1520342        67.25
5           mths_since_last_delinq        1158535        51.25
6                          il_util        1068883        47.28
7               mths_since_rcnt_il         909957        40.25
8                         all_util         866381        38.32
9                      open_acc_6m         866163        38.31
10                    total_bal_il         866162        38.31
11                     open_rv_12m         866162        38.31
12                     open_rv_24m         866162        38.31
13                      max_bal_bc         866162        38.31
14                      

In [74]:
missing_analysis[
    (missing_analysis["missing_pct"] > 30) &
    (missing_analysis["missing_pct"] <= 50)
]

,feature,missing_count,missing_pct
6,il_util,1068883,47.28
7,mths_since_rcnt_il,909957,40.25
8,all_util,866381,38.32
9,open_acc_6m,866163,38.31
10,total_bal_il,866162,38.31
11,open_rv_12m,866162,38.31
12,open_rv_24m,866162,38.31
13,max_bal_bc,866162,38.31
14,inq_fi,866162,38.31
15,total_cu_tl,866163,38.31


In [75]:
missing_analysis[
    missing_analysis["missing_pct"] > 50
]

,feature,missing_count,missing_pct
0,desc,2134636,94.42
1,mths_since_last_record,1901545,84.11
2,mths_since_recent_bc_dlq,1741000,77.01
3,mths_since_last_major_derog,1679926,74.31
4,mths_since_recent_revol_delinq,1520342,67.25
5,mths_since_last_delinq,1158535,51.25


## 9.4 Temporal Missingness in Credit-Bureau Features

A group of candidate variables exhibits nearly identical missingness of approximately 38.31%. The consistency of this missingness pattern across multiple related credit attributes suggests that the absence may be systematic rather than independent missing data.

To investigate whether this pattern reflects changes in data availability over time, missingness is examined against the loan issue date. The `issue_d` variable is used only for this diagnostic analysis and remains excluded from the predictive feature set.

In [76]:
# Convert issue date for diagnostic analysis only
issue_date = pd.to_datetime(
    df["issue_d"],
    format="%b-%Y",
    errors="coerce"
)

df["issue_year_analysis"] = issue_date.dt.year

In [77]:
year_missingness = (
    df.groupby("issue_year_analysis")["open_acc_6m"]
      .apply(lambda x: x.isna().mean() * 100)
      .round(2)
)

year_missingness

,open_acc_6m
issue_year_analysis,
2007.0,100.00
2008.0,100.00
2009.0,100.00
2010.0,100.00
2011.0,100.00
2012.0,100.00
2013.0,100.00
2014.0,100.00
2015.0,94.92


In [78]:
year_diagnostic = pd.DataFrame({
    "loan_count": df.groupby("issue_year_analysis").size(),
    "open_acc_6m_missing_pct":
        df.groupby("issue_year_analysis")["open_acc_6m"]
          .apply(lambda x: x.isna().mean() * 100)
          .round(2)
})

year_diagnostic

,loan_count,open_acc_6m_missing_pct
issue_year_analysis,,
2007.0,603,100.00
2008.0,2393,100.00
2009.0,5281,100.00
2010.0,12537,100.00
2011.0,21721,100.00
2012.0,53367,100.00
2013.0,134814,100.00
2014.0,235629,100.00
2015.0,421095,94.92


In [79]:
# Analyze all the feature variables with 38.31% missingness
temporal_features = [
    "all_util",
    "open_acc_6m",
    "total_bal_il",
    "open_rv_12m",
    "open_rv_24m",
    "max_bal_bc",
    "inq_fi",
    "total_cu_tl",
    "inq_last_12m",
    "open_il_12m",
    "open_il_24m",
    "open_act_il"
]

temporal_missingness = (
    df.groupby("issue_year_analysis")[temporal_features]
      .agg(lambda x: round(x.isna().mean() * 100, 2))
)

temporal_missingness

,all_util,open_acc_6m,total_bal_il,open_rv_12m,open_rv_24m,max_bal_bc,inq_fi,total_cu_tl,inq_last_12m,open_il_12m,open_il_24m,open_act_il
issue_year_analysis,,,,,,,,,,,,
2007.0,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2008.0,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2009.0,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2010.0,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2011.0,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2012.0,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2013.0,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2014.0,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
2015.0,94.92,94.92,94.92,94.92,94.92,94.92,94.92,94.92,94.92,94.92,94.92,94.92


In [80]:
feature_inventory.loc[
    feature_inventory["feature"].isin(temporal_features),
    "decision"
] = "Remove"

In [81]:
feature_inventory["decision"].value_counts()

,count
decision,
Remove,83
Keep,67
Target,1


### Decision: Temporally Unavailable Credit-Bureau Features

Twelve credit-bureau variables exhibiting approximately 38.31% overall missingness were investigated for temporal patterns in data availability.

The analysis showed a highly systematic pattern: all twelve variables were completely unavailable for loans issued between 2007 and 2014, approximately 94.92% unavailable in 2015, and almost completely available from 2016 onwards.

This indicates that the observed missingness is primarily associated with changes in feature availability or data collection over time rather than borrower-level missing information.

The twelve variables were therefore excluded from the primary modelling feature set. Retaining them across the full study period would require extensive imputation for earlier loan vintages where the information was effectively unavailable and could allow the model to exploit temporal data-availability patterns rather than borrower-specific credit-risk information.

The variables were removed on the basis of temporal consistency and comparability across the study population, rather than by applying an arbitrary missingness threshold.

In [82]:
# Let's analyze missingness of other 2 features il_util and mths_since_rcnt_il
# If the pattern for those same like the 12 features that we removed?

remaining_temporal_check = (
    df.groupby("issue_year_analysis")[
        ["il_util", "mths_since_rcnt_il"]
    ]
    .agg(lambda x: round(x.isna().mean() * 100, 2))
)

remaining_temporal_check

,il_util,mths_since_rcnt_il
issue_year_analysis,,
2007.0,100.00,100.00
2008.0,100.00,100.00
2009.0,100.00,100.00
2010.0,100.00,100.00
2011.0,100.00,100.00
2012.0,100.00,100.00
2013.0,100.00,100.00
2014.0,100.00,100.00
2015.0,95.58,95.06


In [83]:
temporal_features_additional = [
    "il_util",
    "mths_since_rcnt_il"
]

feature_inventory.loc[
    feature_inventory["feature"].isin(temporal_features_additional),
    "decision"
] = "Remove"

In [84]:
feature_inventory["decision"].value_counts()

,count
decision,
Remove,85
Keep,65
Target,1


### Additional Temporal Feature Decision

The remaining two variables in the 30–50% missingness group, `il_util` and `mths_since_rcnt_il`, were also examined by loan issue year.

Both variables exhibited the same broad temporal availability pattern identified in the preceding credit-bureau feature group: they were completely unavailable from 2007 to 2014 and approximately 95% unavailable in 2015. Availability increased substantially from 2016 onwards, although `il_util` continued to exhibit approximately 13–16% missingness during 2016–2018.

Because these variables were effectively unavailable for the earlier loan vintages, they were excluded from the primary modelling feature set to maintain a more consistent predictor space across the full study period and reduce the possibility of models exploiting changes in data availability over time.

In [85]:
candidate_features = feature_inventory.loc[
    feature_inventory["decision"] == "Keep",
    "feature"
].tolist()

missing_analysis, missing_summary = analyze_missingness(
    df,
    candidate_features
)

print("Current candidate features:", len(candidate_features))
print(missing_summary)

Current candidate features: 65
0% missing     18
>0% to 10%     40
>10% to 30%     1
>30% to 50%     0
>50%            6
dtype: int64


## 9.5 Low-Variance and Cardinality Assessment

Following the missingness analysis, the remaining candidate predictors are examined for low information content.

Features with only one observed value provide no discriminatory information and should not be retained for predictive modelling. Features with highly dominant values are also identified for review rather than automatically removed, because low-frequency variation may still contain meaningful credit-risk information.

Categorical cardinality is additionally examined to identify variables that may require special encoding or further review during preprocessing.

In [86]:
def profile_features(df, features):
    """
    Profile candidate features for cardinality and value concentration.
    """

    results = []

    for col in features:
        non_null = df[col].dropna()

        unique_count = non_null.nunique()

        if len(non_null) > 0:
            dominant_pct = (
                non_null.value_counts(normalize=True).iloc[0] * 100
            )
        else:
            dominant_pct = 100.0

        results.append({
            "feature": col,
            "dtype": str(df[col].dtype),
            "unique_values": unique_count,
            "dominant_value_pct": round(dominant_pct, 2)
        })

    return pd.DataFrame(results)

In [87]:
feature_profile = profile_features(
    df,
    candidate_features
)

feature_profile.sort_values(
    ["unique_values", "dominant_value_pct"],
    ascending=[True, False]
).head(30)

,feature,dtype,unique_values,dominant_value_pct
23,application_type,object,2,94.66
1,term,object,2,71.21
5,verification_status,object,3,39.20
54,num_tl_30dpd,float64,5,99.73
3,home_ownership,object,6,49.16
53,num_tl_120dpd_2m,float64,7,99.94
24,acc_now_delinq,float64,9,99.61
32,chargeoff_within_12_mths,float64,11,99.24
2,emp_length,object,11,35.39
59,pub_rec_bankruptcies,float64,12,87.96


In [88]:
constant_features = feature_profile[
    feature_profile["unique_values"] <= 1
]

constant_features

,feature,dtype,unique_values,dominant_value_pct


In [89]:
highly_dominant_features = feature_profile[
    (feature_profile["unique_values"] > 1) &
    (feature_profile["dominant_value_pct"] >= 99)
].sort_values(
    "dominant_value_pct",
    ascending=False
)

highly_dominant_features

,feature,dtype,unique_values,dominant_value_pct
53,num_tl_120dpd_2m,float64,7,99.94
54,num_tl_30dpd,float64,5,99.73
33,delinq_amnt,float64,2617,99.68
24,acc_now_delinq,float64,9,99.61
32,chargeoff_within_12_mths,float64,11,99.24


In [90]:
categorical_profile = feature_profile[
    feature_profile["dtype"] == "object"
].sort_values(
    "unique_values",
    ascending=False
)

categorical_profile

,feature,dtype,unique_values,dominant_value_pct
6,desc,object,124500,0.20
10,earliest_cr_line,object,754,0.68
7,purpose,object,14,56.53
2,emp_length,object,11,35.39
3,home_ownership,object,6,49.16
5,verification_status,object,3,39.20
1,term,object,2,71.21
23,application_type,object,2,94.66


### Low-Variance Assessment Findings

No constant predictors were identified among the retained candidate features.

Five numerical credit-history variables exhibited highly concentrated distributions, with at least 99% of their observed values represented by a single value. These variables describe relatively rare adverse credit events, including recent delinquency and charge-off activity.

They were therefore not excluded solely on the basis of low variance. Although uncommon, such events may carry disproportionate credit-risk information. Their predictive contribution will instead be evaluated during subsequent modelling and feature analysis.

### Categorical Cardinality Findings

The categorical cardinality assessment identified two variables requiring special treatment.

`desc` contains 124,500 unique values and represents unstructured borrower-written text rather than a conventional categorical predictor. It will therefore be processed separately as the textual modality of the hybrid modelling framework.

`earliest_cr_line` contains 754 unique values and represents temporal information rather than a nominal category. Rather than applying high-dimensional categorical encoding, it will subsequently be transformed into a credit-history duration feature relative to the loan issue date.

The remaining categorical predictors exhibit low to moderate cardinality and are suitable for conventional categorical preprocessing.

## 9.6 Numerical Redundancy Assessment

The retained numerical predictors are screened for strong pairwise correlations to identify potentially redundant representations of the same underlying borrower characteristic.

Correlation is used as a diagnostic screening tool rather than an automatic feature-removal criterion. Highly correlated pairs are subsequently reviewed in terms of their definitions and modelling relevance before any exclusion decision is made.

In [91]:
numeric_features = [
    col for col in candidate_features
    if pd.api.types.is_numeric_dtype(df[col])
]

corr_matrix = df[numeric_features].corr()

high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i):
        corr_value = corr_matrix.iloc[i, j]

        if abs(corr_value) >= 0.90:
            high_corr_pairs.append({
                "feature_1": corr_matrix.columns[i],
                "feature_2": corr_matrix.columns[j],
                "correlation": round(corr_value, 4)
            })

high_corr_pairs = (
    pd.DataFrame(high_corr_pairs)
    .sort_values(
        "correlation",
        key=abs,
        ascending=False
    )
    .reset_index(drop=True)
)

high_corr_pairs

,feature_1,feature_2,correlation
0,fico_range_high,fico_range_low,1.0000
1,num_sats,open_acc,0.9990
2,num_rev_tl_bal_gt_0,num_actv_rev_tl,0.9836
3,tot_hi_cred_lim,tot_cur_bal,0.9756


### Numerical Redundancy Findings

Pairwise correlation screening identified four numerical feature pairs with absolute Pearson correlation coefficients greater than or equal to 0.90.

The strongest redundancy was observed between `fico_range_low` and `fico_range_high` (r = 1.000), which represent the lower and upper boundaries of the same FICO score range. These variables will subsequently be consolidated into a single FICO midpoint feature.

`num_sats` and `open_acc` were also almost perfectly correlated (r = 0.999), indicating substantial redundancy. `open_acc` was retained as the representative measure, while `num_sats` was excluded.

Although `num_rev_tl_bal_gt_0` and `num_actv_rev_tl` (r = 0.984), and `tot_hi_cred_lim` and `tot_cur_bal` (r = 0.976), were strongly correlated, the respective variables represent distinguishable credit characteristics. They were therefore retained at this stage, allowing their predictive contribution to be assessed during subsequent modelling.

Overall, correlation was used as a redundancy-screening mechanism rather than as an automatic feature-elimination rule.

In [92]:
feature_inventory.loc[
    feature_inventory["feature"] == "num_sats",
    "decision"
] = "Remove"

In [93]:
candidate_features = feature_inventory.loc[
    feature_inventory["decision"] == "Keep",
    "feature"
].tolist()

print("Final candidate feature count:", len(candidate_features))

feature_inventory["decision"].value_counts()

Final candidate feature count: 64


,count
decision,
Remove,86
Keep,64
Target,1


In [94]:
assert len(feature_inventory) == 151

assert feature_inventory["decision"].isna().sum() == 0

assert set(feature_inventory["decision"].unique()) == {
    "Keep", "Remove", "Target"
}

print("Feature inventory validation passed.")

Feature inventory validation passed.


In [95]:
feature_inventory[
    feature_inventory["decision"] == "Target"
]

,feature,dtype,definition,information_timing,decision,justification
16,loan_status,object,Current status of the loan.,Outcome / target,Target,This variable is used to construct the binary ...


## 9.7 Final Candidate Feature Set

The feature audit systematically evaluated the original dataset variables according to their suitability for application-time credit-risk prediction.

Features were excluded where they represented identifiers, post-outcome information, lender-generated or otherwise unsuitable information, substantial temporal availability artefacts, or clear redundancy. Missingness was not treated as an automatic exclusion criterion; instead, its underlying structure and temporal availability were investigated before decisions were made.

No constant predictors were identified. Several highly concentrated credit-history variables were retained because rare adverse credit events may remain informative for default prediction.

Pairwise correlation was used as a redundancy-screening mechanism rather than an automatic elimination rule. `num_sats` was excluded because of its near-perfect redundancy with `open_acc`. Other highly correlated variables were retained where they represented conceptually distinguishable borrower characteristics.

The two original FICO range boundaries were retained temporarily and will subsequently be consolidated into a single FICO midpoint feature during preprocessing.

Following the audit, 64 candidate predictors remain for subsequent preprocessing and model development, together with `loan_status` as the target variable.

The final feature inventory and candidate-feature list were exported to provide a reproducible hand-off to the subsequent preprocessing stage.

## 9.8 Export Outputs for Notebook 3

The final feature inventory and candidate-feature list are exported to the project directory to provide a reproducible handoff to the subsequent preprocessing notebook. The complete inventory preserves the audit trail for all original variables, while the candidate-feature file records the predictors that passed the Notebook 2 audit.


In [96]:
from pathlib import Path

output_dir = Path(
    "/content/drive/MyDrive/Credit_Risk_Thesis/data/processed/notebook_02"
)

output_dir.mkdir(parents=True, exist_ok=True)

feature_inventory.to_csv(
    output_dir / "feature_inventory_final.csv",
    index=False
)

pd.DataFrame({
    "feature": candidate_features
}).to_csv(
    output_dir / "final_candidate_features.csv",
    index=False
)

print("Notebook 2 outputs saved to:", output_dir)


Notebook 2 outputs saved to: /content/drive/MyDrive/Credit_Risk_Thesis/data/processed/notebook_02


# 10. Notebook Summary and Handoff to Preprocessing

## Key Findings

- All 151 original LendingClub variables were systematically audited for suitability for application-time credit-risk prediction.
- Variables representing identifiers, post-origination repayment behaviour, recovery activity, hardship outcomes, settlement information, and other potential sources of target leakage were excluded.
- Missingness was investigated according to its underlying structure rather than by applying a fixed percentage threshold alone.
- Joint-application variables exhibited structural missingness because joint applications constitute only a small proportion of the dataset; joint-specific predictors were excluded from the primary feature set.
- Five delinquency and derogatory-history variables with substantial missingness were retained because the missingness pattern may itself reflect meaningful borrower credit-history characteristics.
- A group of credit-bureau variables was found to be systematically unavailable in earlier loan vintages and was excluded to reduce temporal data-availability artefacts.
- No constant predictors were identified. Rare adverse-credit indicators were retained despite highly concentrated distributions because they may still contain useful risk information.
- Correlation analysis was used as a redundancy-screening mechanism rather than as an automatic feature-elimination rule. `num_sats` was excluded because of its near-perfect redundancy with `open_acc`.
- `fico_range_low` and `fico_range_high` were retained temporarily and will be consolidated into a single representative FICO feature during preprocessing.
- The final audit retained **64 candidate predictors**, excluded **86 variables**, and identified `loan_status` as the single target variable.

## Handoff to Notebook 03

Notebook 03 will transform the audited candidate feature set into modelling-ready structured and textual representations. The next stage will address feature engineering, treatment of remaining missing values, temporal-variable transformation, categorical encoding, numerical preprocessing, preparation of borrower-description text, and construction of reproducible modelling datasets.

The exported `feature_inventory_final.csv` preserves the complete feature-audit trail, while `final_candidate_features.csv` defines the predictor set entering preprocessing.
